# 04 plugins

Question: do the natives of `golem-bridge` and the plugin runtime of `golem-plugin` do what
`okf/bridge.md`, `okf/plugins/index.md` and `okf/plugins/core.md` say, without a JVM.

The native methods of the host class convert their arguments and call one plain function of
`golem_bridge::host` each. `golem-devtool plugins` drives those functions with a script, as the
mod drives the natives on the server thread, so the notebook judges the same path. The notebook
holds the table of natives to the contract, then the order of the plugins, the cancel chain, the
panics, the budgets, the commands with the sources and levels of their senders, the permissions,
the core plugin, the store that the runtime hands the plugins and flushes once the server stopped,
the check of a node for a sender with the reload of the permissions of a store, the plugin
golem-permissions with the PermissionAPI, the operator levels that it keeps, the commands of the
port of LuckPerms and a change that the store refuses, the wire form of the actions, the actions
on players, the game with the guard of the server thread and the wire form of its answers, the
tools of operators, the queue, the tick statistics and the pass values of a broken handler. The
game is a scripted one in place of the one the mod connects. The natives themselves run only
inside a JVM, which the end to end test with the Forge server and the bridge mod covers.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The budget of one plugin call in the scripts, in milliseconds. A test plugin call takes
# microseconds, so a call over it is one that sleeps.
budget_ms = 20

# How long the slow test plugin sleeps, well over the budget.
slow_ms = 60

# The pause between the ticks of the wall clock section, in milliseconds.
tick_pause_ms = 100

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo,
rustc and Golem. The helpers run a subcommand, write a script into a temporary directory and keep
the lines of the log that start with `[golem]`.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-04-"))
WORKSPACE = tomllib.loads((P.root / "Cargo.toml").read_text(encoding="utf-8"))["workspace"]["package"]

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"


def golem_lines(stderr):
    return [line[len("[golem] "):] for line in stderr.splitlines() if line.startswith("[golem] ")]


def devtool(*arguments, expect=0):
    result = run([DEVTOOL, *arguments], cwd=P.root, timeout=300)
    assert result.returncode == expect, "golem-devtool %s exited %d\n--- stdout\n%s--- stderr\n%s" % (
        " ".join(map(str, arguments)), result.returncode, result.stdout, result.stderr)
    return (json.loads(result.stdout) if result.stdout.strip() else None), golem_lines(result.stderr)


def write(name, document):
    path = SCRATCH / name
    path.write_text(json.dumps(document), encoding="utf-8")
    return path


def script(name, document):
    out, log = devtool("plugins", write(name + ".json", document))
    return out["steps"], log


def budgets_of(steps):
    rows = [step for step in steps if step["op"] == "budgets"][-1]["answer"]
    return {row["plugin"]: row for row in rows}


def broadcasts(drain_step):
    return [action["text"] for action in drain_step["decoded"] if action["kind"] == "broadcast"]


ALEX = {"uuid": "6f2d5b3e-1c4a-4e8b-9a7d-2b1c0e9f8a71", "name": "Alex"}
SAM = {"uuid": "0b9a8c7d-6e5f-4a3b-8c2d-1e0f9a8b7c6d", "name": "Sam"}
NETHER = {"dimension": "minecraft:the_nether", "x": 1.5, "y": 64.0, "z": -2.25, "yaw": 90.0, "pitch": -12.5}
NETHER_TEXT = "minecraft:the_nether 1.50 64.00 -2.25, yaw 90.0, pitch -12.5"
LAVA = {"dimension": "minecraft:overworld", "x": -10.0, "y": 11.0, "z": 300.75, "yaw": -45.0, "pitch": 0.0}
LAVA_TEXT = "minecraft:overworld -10.00 11.00 300.75, yaw -45.0, pitch 0.0"

# The permissions of the scripts: Alex is in the group admin, which grants every node of Golem.
PERMISSIONS = '''
[groups.default]
nodes = ["essentials.home"]

[groups.admin]
weight = 100
inherits = ["default"]
nodes = ["golem.*", "-golem.listname"]

[players.%s]
groups = ["admin"]
''' % ALEX["uuid"]
natives_table, _ = devtool("natives")
print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("golem:", natives_table["version"])
print("devtool:", P.rel(DEVTOOL) if P.rel(DEVTOOL) != str(DEVTOOL) else "golem-devtool in the target directory of cargo")

## The natives against the contract

`register_natives` hands the JVM a table of names, descriptors and functions. The cell reads the
host class of `okf/bridge.md`, derives the JNI descriptor of every native method from its Java
declaration and compares both lists name by name, in order. The version that `version()` answers
is the version of the workspace.

In [ ]:
bridge_md = (P.root / "okf" / "bridge.md").read_text(encoding="utf-8")
java = bridge_md.split("```java", 1)[1].split("```", 1)[0]
package = re.search(r"package ([\w.]+);", java).group(1)
klass = re.search(r"public final class (\w+)", java).group(1)
TYPES = {"String": "Ljava/lang/String;", "String[]": "[Ljava/lang/String;", "long": "J", "int": "I", "boolean": "Z",
         "double": "D", "float": "F", "void": "V", "Game": "L%s/Game;" % "com/skytekx/golem/host"}
declared = []
for returns, name, parameters in re.findall(r"public static native ([\w\[\]]+) (\w+)\(([^)]*)\);", java):
    types = [TYPES[" ".join(parameter.split()[:-1])] for parameter in parameters.split(",") if parameter.strip()]
    declared.append((name, "(%s)%s" % ("".join(types), TYPES[returns])))

registered = [(row["name"], row["signature"]) for row in natives_table["natives"]]
for (name, signature), (got_name, got_signature) in zip(declared, registered):
    print("%-15s %-24s %s" % (name, signature if len(signature) < 24 else signature[:21] + "...", "ok" if (name, signature) == (got_name, got_signature) else "WRONG"))
assert natives_table["hostClass"] == (package + "." + klass).replace(".", "/"), natives_table["hostClass"]
assert len(declared) == 18, "okf/bridge.md declares %d natives" % len(declared)
assert registered == declared, "the registered table differs from okf/bridge.md:\n%s\n%s" % (registered, declared)
assert natives_table["version"] == WORKSPACE["version"], natives_table["version"]
steps, _ = script("version", {"steps": [{"op": "version"}]})
assert steps[0]["answer"] == WORKSPACE["version"], steps

## The order of the plugins and the cancel chain

Two echo plugins broadcast the name of every event they see, and between them the plugin censor
cancels a chat message that holds the word spam and answers cancel to every other event. The
runtime must call the plugins in their order, end the chain at the first cancel of a chat
message, answer true for it, and ignore a cancel of an event that cannot be cancelled. A respawn
on the way out of the End must reach the plugins as one, apart from a respawn after a death.

In [ ]:
steps, log = script("order", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "echo", "name": "echo-a"},
        {"kind": "cancel", "name": "censor", "word": "spam"},
        {"kind": "echo", "name": "echo-b"},
    ],
    "steps": [
        {"op": "playerJoined", **ALEX},
        {"op": "drain"},
        {"op": "chat", **ALEX, "message": "hello"},
        {"op": "drain"},
        {"op": "chat", **ALEX, "message": "buy spam now"},
        {"op": "drain"},
        {"op": "respawn", **ALEX, "dimension": "minecraft:overworld"},
        {"op": "drain"},
        {"op": "respawn", **ALEX, "dimension": "minecraft:overworld", "endConquered": True},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] is None, steps[0]
assert broadcasts(steps[1]) == ["echo-a playerJoined", "echo-b playerJoined"], steps[1]
assert steps[2]["answer"] is False, "a chat message without the word was cancelled"
assert broadcasts(steps[3]) == ["echo-a chat", "echo-b chat"], steps[3]
assert steps[4]["answer"] is True, "the chat message with the word was not cancelled"
assert broadcasts(steps[5]) == ["echo-a chat"], "the chain went on after the cancel: %s" % steps[5]
assert broadcasts(steps[7]) == ["echo-a respawn", "echo-b respawn"], steps[7]
assert broadcasts(steps[9]) == ["echo-a respawn from the End", "echo-b respawn from the End"], steps[9]
calls = {name: row["calls"] for name, row in budgets_of(steps).items()}
print("calls:", calls)
assert calls == {"golem-core": 5, "echo-a": 5, "censor": 5, "echo-b": 4}, calls
assert log == [], log

## Panics are caught

The plugin fragile panics on every death and in its command. The runtime must catch each panic,
count it, and go on as if the plugin passed, so the plugin after it still sees the event with the
place of the death. A command whose plugin panics answers one line that says so. The log names the
panics of a plugin the first time, the tenth, the hundredth and at every further power of ten, so
the three panics of fragile give one line. The plugin flood panics on each of 100 ticks, run with
`RUST_BACKTRACE=1`: its budget must count 100 panics, the log must name the first, the tenth and
the hundredth, and the panic hook of the standard library must print no line and no backtrace for
any of them.

In [ ]:
steps, log = script("panics", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "echo", "name": "echo-a"},
        {"kind": "panic", "name": "fragile", "on": "death", "commands": ["fragile"]},
        {"kind": "echo", "name": "echo-b"},
    ],
    "steps": [
        {"op": "death", **ALEX, "cause": "minecraft:fall", **NETHER},
        {"op": "death", **SAM, "cause": "minecraft:lava", **LAVA},
        {"op": "command", "name": "fragile", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "arguments": "now"},
        {"op": "playerLeft", **ALEX},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] is None and steps[1]["answer"] is None, steps[:2]
assert steps[2]["answer"] == ["The command fragile failed, and the log of the server holds the cause."], steps[2]
assert broadcasts(steps[4]) == [
    "echo-a death at " + NETHER_TEXT, "echo-b death at " + NETHER_TEXT,
    "echo-a death at " + LAVA_TEXT, "echo-b death at " + LAVA_TEXT, "echo-a playerLeft", "echo-b playerLeft",
], steps[4]
fragile = budgets_of(steps)["fragile"]
print("fragile:", fragile)
assert fragile["calls"] == 4 and fragile["panics"] == 3, fragile
panics = [line for line in log if line.startswith("the plugin fragile panicked")]
for line in panics:
    print(line)
assert panics == ["the plugin fragile panicked on the event death with \"the test plugin fragile panics on death\", so "
                  "the runtime goes on as if it passed, 1 times so far"], log

flood = subprocess.run([str(DEVTOOL), "plugins", str(write("flood.json", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "panic", "name": "flood", "on": "tick"}],
    "steps": [{"op": "tick", "nanos": 1000000, "repeat": 100}, {"op": "budgets"}],
}))], cwd=P.root, timeout=300, capture_output=True, text=True, env=dict(os.environ, **QUIET, RUST_BACKTRACE="1"))
assert flood.returncode == 0, flood.stderr
rows = budgets_of(json.loads(flood.stdout)["steps"])
print("flood:", rows["flood"], "lines on the standard error:", len(flood.stderr.splitlines()))
assert rows["flood"]["calls"] == 100 and rows["flood"]["panics"] == 100, rows
named = [line for line in golem_lines(flood.stderr) if " panicked " in line]
assert [line.rsplit(", ", 1)[1] for line in named] == ["1 times so far", "10 times so far", "100 times so far"], named
assert not [line for line in flood.stderr.splitlines() if not line.startswith("[golem] ")], flood.stderr[:2000]

## Budgets

The plugin sluggish sleeps longer than the budget on its first two joins. The runtime must time
every call, count both calls over the budget, keep the longest call, and log a call over the
budget at most once a minute per plugin. The budgets are printed as counts, since the times
differ from run to run. The status of the core plugin names the same counts.

In [ ]:
steps, log = script("budgets", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "slow", "name": "sluggish", "on": "playerJoined", "sleepMs": slow_ms, "times": 2},
    ],
    "steps": [
        {"op": "playerJoined", **ALEX},
        {"op": "playerJoined", **SAM},
        {"op": "playerJoined", **ALEX},
        {"op": "tick", "nanos": 1000000, "repeat": 2},
        {"op": "budgets"},
        {"op": "command", "name": "golem", "arguments": "status"},
    ],
})
rows = budgets_of(steps)
print("budgets:", rows)
assert rows["sluggish"] == {"plugin": "sluggish", "calls": 5, "overBudget": 2, "panics": 0, "longestOverBudget": True}, rows
assert rows["golem-core"] == {"plugin": "golem-core", "calls": 5, "overBudget": 0, "panics": 0, "longestOverBudget": False}, rows
warnings = [line for line in log if "over the budget" in line]
print(warnings)
assert len(warnings) == 1 and warnings[0].startswith("the plugin sluggish took ") and "on the event playerJoined" in warnings[0], log
status = steps[5]["answer"]
for line in status:
    print(line)
assert status[1] == "Plugins: 2, with a budget of %.3f ms per call" % budget_ms, status
assert re.fullmatch(r"golem-core: calls 5, total [0-9.]+ ms, longest [0-9.]+ ms, over budget 0, panics 0", status[2]), status
assert re.fullmatch(r"sluggish: calls 5, total [0-9.]+ ms, longest [0-9.]+ ms, over budget 2, panics 0", status[3]), status
longest = float(re.search(r"longest ([0-9.]+) ms", status[3]).group(1))
assert longest >= slow_ms, "the longest call of sluggish took %.3f ms, less than its sleep" % longest

## Commands

The first plugin that declares a command owns it. A later plugin that declares the same name, or
a name that is empty or holds a space, is logged and refused. A command reaches its owner with its
sender, its level and its arguments, and an unknown command answers one line. A UUID and a name
are a player, neither at level 4 is the console, and a name alone is another source at its level,
such as a command block at level 2 or a zombie that runs the command at level 4. A UUID without a
name, neither of them below level 4 and a level outside 0 to 4 are refused rather than run as the
console, and a null `arguments` reads as the empty string.

In [ ]:
steps, log = script("commands", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "echo", "name": "echo-a", "commands": ["echo-a"], "quiet": True},
        {"kind": "echo", "name": "echo-b", "commands": ["golem", "echo-a", "bad name", "", "echo-b"], "quiet": True},
    ],
    "steps": [
        {"op": "commandNames"},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 0,
         "arguments": "a b"},
        {"op": "command", "name": "echo-b", "level": 4, "arguments": ""},
        {"op": "command", "name": "echo-a"},
        {"op": "command", "name": "nope", "arguments": "x"},
        {"op": "command", "name": "echo-a", "senderName": "@", "level": 2, "arguments": "lamp"},
        {"op": "command", "name": "echo-b", "senderName": "Zombie", "level": 4},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "arguments": "x"},
        {"op": "command", "name": "echo-a", "level": 2, "arguments": "x"},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 5},
        {"op": "command", "name": "echo-a", "senderName": "@", "level": -1},
        {"op": "command", "arguments": "x"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] == ["golem", "echo-a", "echo-b"], steps[0]
assert steps[1]["answer"] == ["echo-a ran echo-a for Alex with [a b]"], steps[1]
assert steps[2]["answer"] == ["echo-b ran echo-b for the console with []"], steps[2]
assert steps[3]["answer"] == ["echo-a ran echo-a for the console with []"], steps[3]
assert steps[4]["answer"] == ["The command nope is unknown."], steps[4]
assert steps[5]["answer"] == ["echo-a ran echo-a for @ at level 2 with [lamp]"], steps[5]
assert steps[6]["answer"] == ["echo-b ran echo-b for Zombie at level 4 with []"], steps[6]
assert all(step["answer"] == [] for step in steps[7:12]), steps[7:12]
assert steps[12]["decoded"] == [
    {"kind": "message", "uuid": ALEX["uuid"], "text": "echo-a heard Alex"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-b ran echo-b"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-b ran echo-b"},
], steps[12]
assert budgets_of(steps)["echo-a"]["calls"] == 3 and budgets_of(steps)["echo-b"]["calls"] == 2, budgets_of(steps)
for line in log:
    print(line)
assert log == [
    "the plugin echo-b declares the command golem, which the plugin golem-core owns, so the runtime ignores it",
    "the plugin echo-b declares the command echo-a, which the plugin echo-a owns, so the runtime ignores it",
    "the plugin echo-b declares the command \"bad name\", which is no valid name, so the runtime ignores it",
    "the plugin echo-b declares the command \"\", which is no valid name, so the runtime ignores it",
    "the native command got senderUuid without senderName, so it refuses the command echo-a",
    "the native command got neither senderUuid nor senderName at the level 2, which only the console has at 4, so it refuses the command echo-a",
    "the native command got the level 5, which is no permission level, so it refuses the command echo-a",
    "the native command got the level -1, which is no permission level, so it refuses the command echo-a",
    "the native command got null for name, so it answers the pass value",
], log

## Permissions

The mod asks the native permission for every node of type Boolean of the PermissionAPI of Forge.
The first plugin that answers decides, a plugin that answers nothing leaves the node to the next,
a plugin that panics is caught and counted and the chain goes on, and without any answer the node
falls to its default resolver, which the native answers as none. A null node is logged and
answered as none. The native takes the UUID and the node alone, two strings, so a dynamic context
of Forge, such as the dimension of a check, never reaches a plugin, and the answer of a plugin
holds in every context, as `okf/bridge.md` says.

In [ ]:
assert dict(registered)["permission"] == "(Ljava/lang/String;Ljava/lang/String;)I", "the native takes a context"
steps, log = script("permissions", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "grant", "name": "ranks", "grants": ["forge.use_entity_selectors"], "denies": ["golem.fly"]},
        {"kind": "panic", "name": "fragile", "on": "permission"},
        {"kind": "grant", "name": "late", "grants": ["golem.fly", "golem.late"]},
    ],
    "steps": [
        {"op": "permission", "uuid": ALEX["uuid"], "node": "forge.use_entity_selectors"},
        {"op": "permission", "uuid": ALEX["uuid"], "node": "golem.fly"},
        {"op": "permission", "uuid": SAM["uuid"], "node": "golem.late"},
        {"op": "permission", "uuid": SAM["uuid"], "node": "golem.unknown"},
        {"op": "permission", "uuid": SAM["uuid"]},
        {"op": "budgets"},
    ],
})
answers = [step["answer"] for step in steps[:5]]
print("answers:", answers)
assert answers == [True, False, True, None, None], answers
calls = {name: (row["calls"], row["panics"]) for name, row in budgets_of(steps).items()}
print("calls and panics:", calls)
assert calls == {"golem-core": (4, 0), "ranks": (4, 0), "fragile": (2, 2), "late": (2, 0)}, calls
# The first panic of a process may take longer than the budget, which the log may name too.
log = [line for line in log if " over the budget of " not in line]
for line in log:
    print(line)
# The log names the first panic of fragile, and its second only in the count of its budget.
assert [line.split(" with ")[0] for line in log] == [
    "the plugin fragile panicked on the permission golem.late",
    "the native permission got null for node, so it answers the pass value",
], log

## A port that answers as LuckPerms through Arclight

On the live server Arclight forwards every Boolean node of an online player to LuckPerms, which
answers a node that nothing names false, or true for an operator, so the default resolver of a
mod is never asked, as `okf/plugins/permissions.md` records. A plugin built like that port must
decide such a node itself: true for an operator and false for every other player, where a plugin
that leaves the node alone lets it fall to the default resolver of the mod. A node that the plugin
names still answers as it names it.

In [ ]:
NODES = ["voicechat.speak", "spark", "golem.fly"]
asks = [{"op": "permission", "uuid": who["uuid"], "node": node} for who in (ALEX, SAM) for node in NODES]
forwarded, _ = script("forwarded", {"plugins": [
    {"kind": "grant", "name": "luckperms", "denies": ["golem.fly"], "operators": [ALEX["uuid"].upper()]}],
    "steps": asks})
left, _ = script("left", {"plugins": [{"kind": "grant", "name": "ranks", "denies": ["golem.fly"]}], "steps": asks})
print("as LuckPerms through Arclight:", [step["answer"] for step in forwarded])
print("left to the default resolver: ", [step["answer"] for step in left])
assert [step["answer"] for step in forwarded] == [True, True, False, False, False, False], forwarded
assert [step["answer"] for step in left] == [None, None, False, None, None, False], left

## The core plugin

`/golem source` names the license, the version, the repository and the commit, and links the tree of
that commit, as section 13 of the AGPL asks, or the repository alone while the commit is unknown or
ends in `-dirty`, the mark of a build whose files differ from its commit. `/golem status` names the
TPS, the MSPT and the ticks of the bridge. `/golem` and `/golem help` list the subcommands, and an
unknown subcommand answers an error that names the word as it was typed, in any case, and the help.
The core plugin logs the connection of the bridge once, however often the server starts.

In [ ]:
REPOSITORY = WORKSPACE["repository"]
COMMIT = "4b825dc642cb6eb9a060e54bf8d69288fbee4904"
core = [{"kind": "core"}]
steps, log = script("core", {
    "budgetMs": budget_ms,
    "source": {"version": WORKSPACE["version"], "repository": REPOSITORY, "commit": COMMIT},
    "plugins": core,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "commandNames"},
        {"op": "command", "name": "golem", "arguments": "source"},
        {"op": "command", "name": "golem"},
        {"op": "command", "name": "golem", "arguments": "help"},
        {"op": "command", "name": "golem", "arguments": "FroBnicate now"},
        {"op": "tick", "nanos": 4000000, "repeat": 3},
        {"op": "stats"},
        {"op": "command", "name": "golem", "arguments": "  STATUS  "},
    ],
})
assert log.count("the bridge is connected: Forge 47.4.18, Minecraft 1.20.1") == 1 and len(log) == 1, log
assert steps[2]["answer"] == ["golem"], steps[2]
source = steps[3]["answer"]
for line in source:
    print(line)
assert source[0] == "Golem %s is free software under the GNU Affero General Public License 3.0 only, AGPL-3.0-only." % WORKSPACE["version"], source
assert "Section 13 of the AGPL" in source[1], source
assert source[2:] == ["Repository: " + REPOSITORY, "Commit: " + COMMIT, "Source of this build: %s/tree/%s" % (REPOSITORY, COMMIT)], source
help_lines = steps[4]["answer"]
assert steps[5]["answer"] == help_lines and len(help_lines) == 9, help_lines
assert [line.split(":")[0] for line in help_lines[1:]] == [
    "/golem source", "/golem status", "/golem whoami", "/golem player <name>", "/golem world [<dimension>]",
    "/golem teleport <name> <dimension> <x> <y> <z>", "/golem listname <name> [<text>]", "/golem help"], help_lines
assert steps[6]["answer"] == ["The subcommand FroBnicate of /golem is unknown."] + help_lines, steps[6]
stats = steps[8]["answer"]
assert steps[9]["answer"][0] == "Golem %s: TPS %.2f, MSPT %.2f, ticks %d" % (WORKSPACE["version"], stats["tps"], stats["mspt"], stats["ticks"]), (steps[9], stats)
assert stats["ticks"] == 3 and stats["mspt"] == 4.0, stats

unknown, _ = script("core-unknown", {
    "source": {"version": WORKSPACE["version"], "repository": REPOSITORY + "/", "commit": "unknown"},
    "plugins": core,
    "steps": [{"op": "command", "name": "golem", "arguments": "source"}],
})
assert unknown[0]["answer"][3:] == ["Commit: unknown", "Source of this build: " + REPOSITORY], unknown
# A build whose files differ from its commit has no tree in the repository that holds its source.
dirty, _ = script("core-dirty", {
    "source": {"version": WORKSPACE["version"], "repository": REPOSITORY, "commit": COMMIT + "-dirty"},
    "plugins": core,
    "steps": [{"op": "command", "name": "golem", "arguments": "source"}],
})
assert dirty[0]["answer"][3:] == ["Commit: %s-dirty" % COMMIT, "Source of this build: " + REPOSITORY], dirty
build, _ = script("core-build", {
    "source": "build",
    "plugins": core,
    "steps": [{"op": "command", "name": "golem", "arguments": "source"}],
})
lines = build[0]["answer"]
commit = lines[3].removeprefix("Commit: ")
# Source::of_build reads GOLEM_COMMIT of the build of golem-plugin alone, as okf/plugins/core.md says.
assert commit == (os.environ.get("GOLEM_COMMIT") or "").strip() or "unknown", commit
assert lines[2] == "Repository: " + REPOSITORY, lines
assert lines[4] == "Source of this build: " + (REPOSITORY if commit == "unknown" or commit.endswith("-dirty") else "%s/tree/%s" % (REPOSITORY, commit)), lines
print("the build names the commit", commit)

## The store of the plugins

The program gives the runtime the store of the plugins, and a plugin reaches it through
`Context::store` of its call. The devtool opens the store that the script names, hands it to the
runtime as `golem run` does and closes it after the last step. Its plugin `keeper` sets and reads a
setting of the store through the context. A context that no runtime made holds no store, and
neither does a runtime that the program gave none. Once every plugin saw `ServerStopped`, the
runtime flushes the store: the cell holds the writer still for a second, so a change of the plugin
waits for the writer, and the step `serverStopped` must leave every change in the file before
anything closes the store. The plugin also sets a setting when it sees `ServerStopped` itself,
behind a writer that it holds still for another second, so a flush that ran before the plugins saw
the event would leave that change waiting. The wait counts in the budget of no plugin, which a
budget of a quarter of the pause shows. A second open of the file after the script reads the three
settings.

In [ ]:
KEPT = SCRATCH / "keeper.db"
KEEP = {"op": "command", "name": "keep"}
PAUSE_MS = 1000
keeper = [{"kind": "core"}, {"kind": "keeper", "name": "keeper", "commands": ["keep"]}]
kept, log = script("keeper", {
    "budgetMs": PAUSE_MS / 4,
    "plugins": keeper,
    "store": str(KEPT),
    "steps": [
        {**KEEP, "arguments": "get greeting"},
        {**KEEP, "arguments": "set greeting hello there"},
        {**KEEP, "arguments": "get greeting"},
        {**KEEP, "arguments": "outside"},
        {**KEEP, "arguments": "pause %d" % PAUSE_MS},
        {**KEEP, "arguments": "set farewell so long"},
        {**KEEP, "arguments": "stats"},
        {**KEEP, "arguments": "stopped %d parting" % PAUSE_MS},
        {"op": "serverStopped"},
        {**KEEP, "arguments": "stats"},
        {"op": "budgets"},
    ],
})
said = [step["answer"] for step in kept]
for answer in said[:10]:
    print(answer)
assert said[0] == ["greeting is not set"], said[0]
assert said[1] == ["set greeting"] and said[2] == ["greeting is hello there"], said[1:3]
assert said[3] == ["store false"], "a context that no runtime made holds a store: %s" % said[3]
assert said[4] == ["paused %d ms" % PAUSE_MS] and said[5] == ["set farewell"], said[4:6]
waiting = re.fullmatch(r"queued 2, written ([01]), pending ([12])", said[6][0])
assert waiting and int(waiting[1]) + int(waiting[2]) == 2, "no change waits for the writer: %s" % said[6]
assert said[7] == ["armed parting behind %d ms" % PAUSE_MS], said[7]
# The third change is the one that the plugin made on ServerStopped itself, behind a held writer.
assert said[9] == ["queued 3, written 3, pending 0"], "serverStopped left a change out of the file: %s" % said[9]
rows = budgets_of(kept)
print("budgets:", rows)
for plugin in ("golem-core", "keeper"):
    assert rows[plugin]["overBudget"] == 0 and not rows[plugin]["longestOverBudget"], (
        "the wait for the flush counts in the budget of %s: %s" % (plugin, rows[plugin]))
assert log == [], log

# The store closed after the last step, so a second open reads from the file what the plugin set.
reopen = write("keeper-reopen.json", {"steps": [{"op": "open", "path": str(KEPT)}, {"op": "settings"},
                                                {"op": "close"}]})
done = run([DEVTOOL, "store", reopen], cwd=P.root, timeout=300)
assert done.returncode == 0, done.stderr
reopened = [json.loads(line) for line in done.stdout.splitlines()]
print("the settings in the file:", reopened[1]["ok"])
assert reopened[1]["ok"] == {"farewell": "so long", "greeting": "hello there", "parting": "stopped"}, reopened[1]

# A runtime that the program gave no store hands its plugins none.
bare, log = script("keeper-without-store", {
    "budgetMs": budget_ms,
    "plugins": keeper,
    "steps": [{**KEEP, "arguments": "get greeting"}, {**KEEP, "arguments": "outside"}],
})
assert [step["answer"] for step in bare] == [["no store"], ["store false"]], bare
assert log == [], log

## A store whose writer is behind

The runtime hands its plugins the handle of the store that never waits, so a call of a plugin is
never held by the writer of the store. The cell holds the writer still and lets the plugin `keeper`
set a setting more often than the store lets changes wait, the number that `okf/store.md` names as
`MAX_PENDING`. The plugin must hear the refusal of every change beyond that number with its reason,
which a command can tell its sender, `room` must answer it before the change, the cache must hold
what the store took and nothing else, and no change may have reached the file while the writer stood
still. The log names the refusal once. Once the runtime flushed the store on `ServerStopped`, the
plugin changes the store again, and a second open of the file reads what the store took.

In [ ]:
BEHIND = SCRATCH / "behind.db"
BOUND = int(re.search(r"\| `MAX_PENDING` \| (\d+),", (P.root / "okf" / "store.md").read_text(encoding="utf-8")).group(1))
HELD_MS = 4000
busy = "the store takes no change now, since %d changes wait for its writer" % BOUND
heard, log = script("keeper-behind", {
    "budgetMs": 60000,
    "plugins": keeper,
    "store": str(BEHIND),
    "steps": [
        {**KEEP, "arguments": "room %d" % BOUND},
        {**KEEP, "arguments": "pause %d" % HELD_MS},
        {**KEEP, "arguments": "fill %d" % (BOUND + 500)},
        {**KEEP, "arguments": "room 1"},
        {**KEEP, "arguments": "set late x"},
        {**KEEP, "arguments": "get fill"},
        {**KEEP, "arguments": "stats"},
        {"op": "serverStopped"},
        {**KEEP, "arguments": "room 1"},
        {**KEEP, "arguments": "set later y"},
    ],
})
said = [step["answer"] for step in heard]
for answer in said:
    print(answer)
assert said[0] == ["room for %d: true" % BOUND] and said[1] == ["paused %d ms" % HELD_MS], said[:2]
assert said[2] == ["took %d of %d: %s" % (BOUND, BOUND + 500, busy)], said[2]
assert said[3] == ["room for 1: false"] and said[4] == ["refused late: " + busy], said[3:5]
# The cache holds what the store took and nothing of what it refused.
assert said[5] == ["fill is %d" % (BOUND - 1)], said[5]
assert said[6] == ["queued %d, written 0, pending %d" % (BOUND, BOUND)], "the plugin outran the pause: %s" % said[6]
assert said[8] == ["room for 1: true"] and said[9] == ["set later"], said[8:]
assert len(log) == 1 and log[0].endswith(
    "is %d changes behind, so the store refuses a change that must not wait for it" % BOUND), log

reopen = write("behind-reopen.json", {"steps": [{"op": "open", "path": str(BEHIND)}, {"op": "settings"},
                                                {"op": "close"}]})
done = run([DEVTOOL, "store", reopen], cwd=P.root, timeout=300)
assert done.returncode == 0, done.stderr
reopened = [json.loads(line) for line in done.stdout.splitlines()]
print("the settings in the file:", reopened[1]["ok"])
assert reopened[1]["ok"] == {"fill": str(BOUND - 1), "later": "y"}, reopened[1]

## The check of a node for a sender

A plugin asks the runtime whether a sender holds a node through `Context::check_permission`, with
the rule that decided, and `Context::has_permission`, without it, which must agree. The console
holds every node. Another source holds every node from the level 2 and none below it, also when it
runs a command as Alex through `execute as`, so it never borrows the nodes of a player. A player
holds what `golem-permissions` grants them, with its reason, and a node that no rule names is not
held and not set, so a port whose upstream grants a node by default can tell. A runtime without
permissions grants a player nothing. The plugin `checker` of the devtool answers each node of its
arguments with the check.

In [ ]:
NODES = "golem.player golem.listname essentials.home essentials.fly"
CHECK = {"op": "command", "name": "check", "arguments": NODES}
SENDERS = [
    ("console", {}),
    ("Alex", {"senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 0}),
    ("Sam", {"senderUuid": SAM["uuid"], "senderName": SAM["name"], "level": 4}),
    ("block", {"senderName": "@", "level": 2}),
    ("as Alex", {"senderName": ALEX["name"], "level": 1}),
]
checked, log = script("checks", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS,
    "steps": [{**CHECK, **sender} for _, sender in SENDERS],
})
said = {name: step["answer"] for (name, _), step in zip(SENDERS, checked)}
for name, lines in said.items():
    for line in lines:
        print("%-8s %s" % (name, line))
assert said["console"] == ["%s: granted true, set true, has true, The console holds every node" % node
                           for node in NODES.split()], said["console"]
assert said["Alex"] == [
    "golem.player: granted true, set true, has true, Group admin grants golem.*",
    "golem.listname: granted false, set true, has false, Group admin denies golem.listname",
    "essentials.home: granted true, set true, has true, Group default grants essentials.home",
    "essentials.fly: granted false, set false, has false, No rule names essentials.fly",
], said["Alex"]
assert said["Sam"] == [
    "golem.player: granted false, set false, has false, No rule names golem.player",
    "golem.listname: granted false, set false, has false, No rule names golem.listname",
    "essentials.home: granted true, set true, has true, Group default grants essentials.home",
    "essentials.fly: granted false, set false, has false, No rule names essentials.fly",
], said["Sam"]
assert said["block"] == ["%s: granted true, set true, has true, The source @ runs at the level 2, and a source that "
                         "is no player holds every node from the level 2" % node for node in NODES.split()]
assert said["as Alex"] == ["%s: granted false, set false, has false, The source Alex runs at the level 1, and a "
                           "source that is no player holds no node below the level 2" % node for node in NODES.split()]
bare, _ = script("checks-without-permissions", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "checker", "name": "checker", "commands": ["check"]}],
    "steps": [{**CHECK, **SENDERS[1][1]}],
})
assert bare[0]["answer"][0] == ("golem.player: granted false, set false, has false, The runtime holds no permissions, "
                                "so no rule names golem.player"), bare[0]
assert log == [], log

## The permissions of a store

With a store the runtime holds the resolver over the set of the store, which the devtool seeds as
`golem run` does. A change that another writer puts into the store behind the resolver moves the
generation of the store, and the resolver keeps its set until the step `refresh` wakes its reload,
which loads the set of that generation on the thread `golem-permissions`, so a check then sees the
change. A set that `golem-permissions` refuses, such as one whose inheritance is a cycle, is
logged once, also when a second `refresh` finds the same generation, and the checks keep the set
they hold. The store closes after the last step.

In [ ]:
STORE = SCRATCH / "permissions.db"
CHECK_SAM = {**CHECK, **SENDERS[2][1], "arguments": "essentials.fly golem.player"}
reloaded, log = script("reload", {
    "plugins": [{"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS,
    "store": str(STORE),
    "steps": [
        CHECK_SAM,
        {"op": "generation"},
        {"op": "behind", "group": "default", "nodes": ["essentials.home", "essentials.fly"]},
        {"op": "generation"},
        CHECK_SAM,
        {"op": "refresh"},
        CHECK_SAM,
        {"op": "behind", "group": "default", "inherits": ["admin"], "nodes": ["essentials.home"]},
        {"op": "refresh"},
        {"op": "generation"},
        CHECK_SAM,
        {"op": "refresh"},
        CHECK_SAM,
    ],
})
answers = [step["answer"] for step in reloaded]
for answer in answers:
    print(answer)
FLY_UNSET = "essentials.fly: granted false, set false, has false, No rule names essentials.fly"
FLY_SET = "essentials.fly: granted true, set true, has true, Group default grants essentials.fly"
assert answers[0][0] == FLY_UNSET, answers[0]
assert answers[1] == {"resolver": 1, "store": 1}, answers[1]
assert answers[2] is True and answers[3] == {"resolver": 1, "store": 2}, answers[2:4]
assert answers[4][0] == FLY_UNSET, "the resolver loaded the set on the server thread: %s" % answers[4]
assert answers[5] == 2 and answers[6][0] == FLY_SET, answers[5:7]
assert answers[7] is True and answers[8] == 2 and answers[9] == {"resolver": 2, "store": 3}, answers[7:10]
assert answers[10][0] == FLY_SET, "the checks lost the set they held: %s" % answers[10]
# A second refresh finds the generation that did not load, so it must neither load nor log it again.
assert answers[11] == 2 and answers[12][0] == FLY_SET, answers[11:13]
cycle = [line for line in log if "do not load" in line]
print(cycle)
assert cycle == ["the permissions of the store %s at the generation 3 do not load, so the checks keep the "
                 "permissions they hold: groups.default.inherits: the inheritance admin, default, admin is a cycle"
                 % STORE], log

## The wire form of the actions

Every action is one string, its fields joined by U+001F, the first field its kind. A field that
holds U+001F has it replaced by a space, and the first replacement of a process is logged, later
ones are not. A teleport carries its id first and the radius of its search for a safe place
last, empty for a teleport to the place as it is and held to 8 blocks. A number of a teleport is
the shortest decimal that reads back as the same number, an infinity is spelt as Java spells it,
`Infinity`, which `Double.parseDouble` reads, and not as `Display` of Rust writes it, `inf`, and an
empty list name clears the name. A line of an unknown kind, with a wrong number of fields or with a
number that does not read decodes to nothing.

In [ ]:
US = "\x1f"
TELEPORT = {"kind": "teleport", "id": 7, "uuid": ALEX["uuid"], **NETHER, "safe": None}
entries = [
    {"kind": "command", "command": "save-all flush"},
    {"kind": "message", "uuid": ALEX["uuid"], "text": "Welcome back"},
    {"kind": "broadcast", "text": "The server restarts in 5 minutes."},
    TELEPORT,
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
    {"kind": "message", "uuid": ALEX["uuid"], "text": "a" + US + "b"},
    {"kind": "broadcast", "text": US + "x" + US},
    {"kind": "command", "command": "say " + US},
    {"kind": "teleport", "id": 9223372036854775807, "uuid": ALEX["uuid"], "dimension": "minecraft:overworld", "x": 0.1,
     "y": -64.0, "z": 1e-07, "yaw": 359.5, "pitch": -90.0, "safe": 3},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": ""},
    {"wire": "explode" + US + ALEX["uuid"]},
    {"wire": "message" + US + ALEX["uuid"]},
    {"wire": "broadcast" + US + "a" + US + "b"},
    {"wire": US.join(["teleport", "1", ALEX["uuid"], "minecraft:overworld", "1", "2", "3", "0", "0"])},
    {"wire": US.join(["teleport", "1", ALEX["uuid"], "minecraft:overworld", "1", "two", "3", "0", "0", ""])},
    {"wire": "listName" + US + ALEX["uuid"]},
    {"wire": "broadcast" + US},
    {"infinite": {"uuid": ALEX["uuid"], "dimension": "minecraft:overworld"}},
    {"kind": "teleport", "id": 9, "uuid": ALEX["uuid"], **NETHER, "safe": 20},
    {"wire": US.join(["teleport", "-1", ALEX["uuid"], "minecraft:overworld", "1", "2", "3", "0", "0", ""])},
    {"wire": US.join(["teleport", "1", ALEX["uuid"], "minecraft:overworld", "1", "2", "3", "0", "0", "-1"])},
]
rows, log = devtool("actions-encode", write("actions.json", entries))
for row in rows:
    print(repr(row["wire"].replace(US, "|")), row["decoded"])
assert rows[0]["wire"] == "command" + US + "save-all flush", rows[0]
assert rows[1]["wire"] == "message" + US + ALEX["uuid"] + US + "Welcome back", rows[1]
assert rows[2]["wire"] == "broadcast" + US + "The server restarts in 5 minutes.", rows[2]
assert rows[3]["fields"] == ["teleport", "7", ALEX["uuid"], "minecraft:the_nether", "1.5", "64", "-2.25", "90", "-12.5",
                             ""], rows[3]
assert rows[4]["fields"] == ["listName", ALEX["uuid"], "[AFK] Alex"], rows[4]
assert rows[5]["fields"] == ["listName", ALEX["uuid"], ""], rows[5]
for row, entry in zip(rows[:6], entries[:6]):
    assert row["decoded"] == entry and row["reencoded"] == row["wire"], row
assert rows[6]["fields"] == ["message", ALEX["uuid"], "a b"], rows[6]
assert rows[7]["fields"] == ["broadcast", " x "], rows[7]
assert rows[8]["fields"] == ["command", "say  "], rows[8]
assert rows[9]["fields"][1] == "9223372036854775807" and rows[9]["fields"][9] == "3", rows[9]
assert rows[9]["fields"][4:9] == ["0.1", "-64", "0.0000001", "359.5", "-90"], rows[9]
assert rows[9]["decoded"] == entries[9], rows[9]
assert rows[10]["fields"] == rows[5]["fields"] and rows[10]["decoded"]["name"] is None, rows[10]
assert [row["decoded"] for row in rows[11:17]] == [None] * 6, rows[11:17]
assert rows[17]["decoded"] == {"kind": "broadcast", "text": ""}, rows[17]
assert rows[18]["fields"] == ["teleport", "1", ALEX["uuid"], "minecraft:overworld", "Infinity", "-Infinity", "0",
                              "Infinity", "NaN", ""], rows[18]
assert rows[18]["decoded"] is not None and rows[18]["reencoded"] == rows[18]["wire"], rows[18]
assert rows[19]["fields"][9] == "8" and rows[19]["decoded"]["safe"] == 8, "the radius is not held to 8: %s" % rows[19]
assert rows[20]["decoded"] is None and rows[21]["decoded"] is None, "an id or a radius below 0 reads: %s" % rows[20:22]
assert log == ["a field of a message action holds U+001F, which the wire form reserves, so it is sent as a space, and later fields like it are not logged again"], log

## The actions on players

The plugin mover queues its actions through the helpers of the context. A teleport goes into the
queue with its place as it is and the next id of the process, a place whose coordinate is no number
and a place at x 30000001, beyond the bounds of the world that `tp` takes, are refused with one
line on the log each and queue nothing and take no id, a teleport with a radius answers its id and
carries the radius, held to 8 blocks, and a list name is set and cleared.

In [ ]:
steps, log = script("mover", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "mover", "name": "mover", "commands": ["move"], "to": NETHER}],
    "steps": [
        {"op": "command", "name": "move", "arguments": "teleport " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "nowhere " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "far " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "name " + ALEX["uuid"] + " [AFK] Alex"},
        {"op": "command", "name": "move", "arguments": "clear " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "safe " + ALEX["uuid"] + " 3"},
        {"op": "command", "name": "move", "arguments": "safe " + ALEX["uuid"] + " 20"},
        {"op": "drain"},
    ],
})
assert [step["answer"] for step in steps[:7]] == [["queued true"], ["queued false"], ["queued false"], ["named"],
                                               ["cleared"], ["queued the teleport 2"], ["queued the teleport 3"]], steps
assert steps[7]["decoded"] == [
    {**TELEPORT, "id": 1},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
    {**TELEPORT, "id": 2, "safe": 3},
    {**TELEPORT, "id": 3, "safe": 8},
], steps[7]
print(log)
assert log == ["a plugin asked to teleport %s to minecraft:the_nether NaN 64.00 -2.25, yaw 90.0, pitch -12.5, which is no finite "
               "place, so the runtime refuses it" % ALEX["uuid"],
               "a plugin asked to teleport %s to minecraft:the_nether 30000001.00 64.00 -2.25, yaw 90.0, pitch -12.5, which "
               "lies outside the bounds of the world, so the runtime refuses it" % ALEX["uuid"]], log

## The end of a teleport

The mod reports the end of every teleport with its id: `teleported` with where the player was and
where they are, each in the wire form of the answer of `location`, or `teleportFailed` with its
reason. Both reach every plugin in its order as an event that cannot be cancelled. Every reason of
the contract reads as itself, and a reason that this build does not know reads as `error` with one
line on the log, so the plugin still learns that the teleport ended. A place that does not read and
a null where the contract names a string are faults of the mod, which are logged, and such a report
ends the teleport of its id with `error`, under the UUID of the report or an empty one, so the id
still gets its one report. An id below 0, which no action carries, reports nothing.

In [ ]:
FROM = {"dimension": "minecraft:overworld", "x": 0.5, "y": 64.0, "z": 0.5, "yaw": 0.0, "pitch": 0.0}
REASONS = ["offline", "dimension", "place", "cancelled", "unsafe", "unloaded", "stopped", "error"]
steps, log = script("teleported", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "echo", "name": "echo-a"}, {"kind": "echo", "name": "echo-b"}],
    "steps": [
        {"op": "teleported", "id": 4, "uuid": ALEX["uuid"], "from": FROM, "to": NETHER},
        {"op": "teleported", "id": 5, "uuid": ALEX["uuid"], "from": US.join(["minecraft:overworld", "1", "2", "3",
                                                                              "4.5", "-90.0"]), "to": NETHER},
        *({"op": "teleportFailed", "id": 10 + index, "uuid": ALEX["uuid"], "reason": reason}
          for index, reason in enumerate(REASONS)),
        {"op": "teleportFailed", "id": 20, "uuid": ALEX["uuid"], "reason": "lost"},
        {"op": "drain"},
        {"op": "teleported", "id": -1, "uuid": ALEX["uuid"], "from": FROM, "to": NETHER},
        {"op": "teleported", "id": 6, "uuid": ALEX["uuid"], "from": "minecraft:overworld" + US + "1", "to": NETHER},
        {"op": "teleported", "id": 7, "uuid": ALEX["uuid"], "from": FROM},
        {"op": "teleportFailed", "id": -2, "uuid": ALEX["uuid"], "reason": "unsafe"},
        {"op": "teleportFailed", "id": 8, "reason": "unsafe"},
        {"op": "drain"},
    ],
})
FROM_TEXT = "minecraft:overworld 0.50 64.00 0.50, yaw 0.0, pitch 0.0"
expected = []
for line in (["teleported 4 %s from %s to %s" % (ALEX["uuid"], FROM_TEXT, NETHER_TEXT)],
             ["teleported 5 %s from minecraft:overworld 1.00 2.00 3.00, yaw 4.5, pitch -90.0 to %s" % (
                 ALEX["uuid"], NETHER_TEXT)],
             *(["teleportFailed %d %s %s" % (10 + index, ALEX["uuid"], reason)] for index, reason in enumerate(REASONS)),
             ["teleportFailed 20 %s error" % ALEX["uuid"]]):
    expected += ["echo-a " + line[0], "echo-b " + line[0]]
print("\n".join(broadcasts(steps[11])))
assert broadcasts(steps[11]) == expected, broadcasts(steps[11])
# A report that does not read ends its id with error, under its UUID or an empty one, and an id below 0
# reports nothing.
unread = []
for line in ("teleportFailed 6 %s error" % ALEX["uuid"], "teleportFailed 7 %s error" % ALEX["uuid"],
             "teleportFailed 8  error"):
    unread += ["echo-a " + line, "echo-b " + line]
print("\n".join(broadcasts(steps[17])))
assert broadcasts(steps[17]) == unread, "the reports that do not read: %s" % steps[17]
print(log)
assert log == [
    'the native teleportFailed got the reason "lost", which this build does not know, so it reports an error',
    "the native teleported got an id below 0, which no teleport carries, so it reports nothing",
    'the native teleported got "minecraft:overworld\\u{1f}1" for from, which is no place',
    "the native teleported got a report of the teleport 6 that does not read, so it reports an error for it",
    "the native teleported got null for to, so it answers the pass value",
    "the native teleported got a report of the teleport 7 that does not read, so it reports an error for it",
    "the native teleportFailed got an id below 0, which no teleport carries, so it reports nothing",
    "the native teleportFailed got null for uuid, so it answers the pass value",
    "the native teleportFailed got a report of the teleport 8 that does not read, so it reports an error for it",
], log

## The teleports that wait at the stop

No tick drains the queue from `serverStopping` on, so a teleport that still waits in it would never
report its end. The natives end each of them with `stopped` inside `serverStopping`, oldest first
and before the plugins hear of the stop, and leave the other actions in the queue. From there to
the next `serverStarting` the context queues no teleport and takes no id, with one line on the log
for each call, and behind it the next teleport takes the next id.

In [ ]:
MOVE = {"op": "command", "name": "move"}
steps, log = script("teleports-at-the-stop", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "echo", "name": "echo-a"},
                {"kind": "mover", "name": "mover", "commands": ["move"], "to": NETHER}],
    "steps": [
        {**MOVE, "arguments": "teleport " + ALEX["uuid"]},
        {"op": "drain"},
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {**MOVE, "arguments": "name " + ALEX["uuid"] + " [AFK] Alex"},
        {**MOVE, "arguments": "teleport " + SAM["uuid"]},
        {"op": "serverStopping"},
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {"op": "serverStopped"},
        {**MOVE, "arguments": "teleport " + ALEX["uuid"]},
        {"op": "drain"},
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {"op": "drain"},
    ],
})
answers = [step["answer"] for step in steps]
assert steps[1]["decoded"] == [{**TELEPORT, "id": 1}], steps[1]
assert answers[2:5] == [["queued the teleport 2"], ["named"], ["queued true"]], answers[2:5]
assert answers[6] == ["refused"] and answers[8] == ["queued false"], "the queue took a teleport at the stop: %s" % answers[6:9]
# The two teleports that waited end with stopped, in their order, before the plugins hear of the stop.
at_stop = steps[9]["decoded"]
print("\n".join(str(action) for action in at_stop))
assert at_stop == [
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "broadcast", "text": "echo-a teleportFailed 2 %s stopped" % ALEX["uuid"]},
    {"kind": "broadcast", "text": "echo-a teleportFailed 3 %s stopped" % SAM["uuid"]},
    {"kind": "broadcast", "text": "echo-a serverStopping"},
    {"kind": "broadcast", "text": "echo-a serverStopped"},
], at_stop
# The refused calls took no id, and the queue takes teleports again once a server starts.
assert answers[11] == ["queued the teleport 4"], answers[11]
assert steps[12]["decoded"] == [{"kind": "broadcast", "text": "echo-a serverStarting"},
                                {**TELEPORT, "id": 4, "safe": 3}], steps[12]
assert log == ["a plugin asked to teleport %s while the server stops, so the runtime refuses it" % ALEX["uuid"]] * 2, log

## A stop that the game does not announce

A server whose loop ends by a fault stops without `serverStopping`: the game still closes its
worlds, and `serverStopped` is the first that Golem hears of the stop. The natives must then do
there what they do at a stop that the game announced: end every teleport that still waits in the
queue with `stopped`, oldest first and before the plugins hear that the server stopped, hand none of
them out, and queue no teleport and take no id until the next `serverStarting`.

In [ ]:
steps, log = script("teleports-at-a-stop-without-notice", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "echo", "name": "echo-a"},
                {"kind": "mover", "name": "mover", "commands": ["move"], "to": NETHER}],
    "steps": [
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {**MOVE, "arguments": "teleport " + SAM["uuid"]},
        {"op": "serverStopped"},
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {"op": "drain"},
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {**MOVE, "arguments": "safe " + ALEX["uuid"] + " 3"},
        {"op": "drain"},
    ],
})
answers = [step["answer"] for step in steps]
assert answers[:2] == [["queued the teleport 1"], ["queued true"]], answers[:2]
assert answers[3] == ["refused"], "the queue took a teleport behind a stop that the game did not announce: %s" % answers[3]
# Both teleports end with stopped, in their order, before the plugins hear that the server stopped, and the
# drain hands out no teleport.
at_stop = steps[4]["decoded"]
print("\n".join(str(action) for action in at_stop))
assert at_stop == [
    {"kind": "broadcast", "text": "echo-a teleportFailed 1 %s stopped" % ALEX["uuid"]},
    {"kind": "broadcast", "text": "echo-a teleportFailed 2 %s stopped" % SAM["uuid"]},
    {"kind": "broadcast", "text": "echo-a serverStopped"},
], at_stop
# The refused call took no id, and the queue takes teleports again once a server starts.
assert answers[6] == ["queued the teleport 3"], answers[6]
assert steps[7]["decoded"] == [{"kind": "broadcast", "text": "echo-a serverStarting"},
                               {**TELEPORT, "id": 3, "safe": 3}], steps[7]
assert log == ["a plugin asked to teleport %s while the server stops, so the runtime refuses it" % ALEX["uuid"]], log

## Styled text

A plugin builds a styled text with `golem_bridge::Text`, and the action carries its JSON, which
`Component.Serializer` of the game reads. `golem-devtool text-json` builds texts from a description
and prints the JSON they write, which must be exactly the JSON of `okf/bridge.md`: the words of a
plugin as they are, newlines included, a colour by name or as `#rrggbb`, the five decorations, the
four clicks, a hover with its own text, an insertion, a translation with its arguments and the texts
after a text in `extra`. What a player wrote or named loses every `§` with the character after it
and every control character, a newline and U+001F among them, and stays a text of its own, so a
player can neither colour the rest of a line nor start one. The JSON escapes the quote, the
backslash, every control character and U+2028, so it holds no U+001F, reads back as JSON, and counts
its length in UTF-16 units. A colour that is neither a name of the game nor six hexadecimal digits
reads as none.

In [ ]:
US = "\x1f"
texts = [
    {"plain": "Teleporting to ", "color": "#ffaa00", "extra": [
        {"player": "home", "color": "#ff5555"}, {"plain": "."}]},
    {"player": "\u00a7cred\u00a7l bold\nline" + US + "end\u007f\u00a7"},
    {"plain": "two\nlines \"quoted\" back\\slash" + US + " tab\t\u2028"},
    {"plain": "all", "bold": True, "italic": False, "underlined": True, "strikethrough": False, "obfuscated": True,
     "color": "dark_purple"},
    {"plain": "open", "click": {"action": "open_url", "value": "https://example.org/golem"}},
    {"plain": "run", "click": {"action": "run_command", "value": "/golem source"}},
    {"plain": "suggest", "click": {"action": "suggest_command", "value": "/msg "}},
    {"plain": "copy", "click": {"action": "copy_to_clipboard", "value": "4000 128 4000"}},
    {"plain": "hover", "hover": {"plain": "the text of the hover", "color": "gray"}, "insertion": "Alex"},
    {"translate": "death.attack.generic", "with": [{"player": "Alex"}], "color": "red"},
    {"plain": "\u00e4\u00f6\u00fc \u2603 \U0001f600"},
    {"strip": "\u00a7\u00a7a\u00a7x\u00a7"},
    {"strip": "plain words stay"},
]
rows, log = devtool("text-json", write("texts.json", texts))
out = rows["texts"]
for row in out:
    print(row)
assert (rows["maxJson"], rows["maxDepth"]) == (262144, 32), rows
assert out[0]["json"] == ('{"text":"Teleporting to ","color":"#ffaa00","extra":[{"text":"home","color":"#ff5555"},'
                          '{"text":"."}]}'), out[0]
assert out[0]["plain"] == "Teleporting to home." and out[0]["depth"] == 2, out[0]
assert out[1]["json"] == '{"text":"red boldlineend"}', "the text of a player keeps a code: %s" % out[1]
assert out[2]["json"] == '{"text":"two\\nlines \\"quoted\\" back\\\\slash\\u001f tab\\t\\u2028"}', out[2]
assert US not in out[2]["json"] and out[2]["parsed"]["text"] == texts[2]["plain"], out[2]
assert out[3]["json"] == ('{"text":"all","color":"dark_purple","bold":true,"italic":false,"underlined":true,'
                          '"strikethrough":false,"obfuscated":true}'), out[3]
assert [row["parsed"]["clickEvent"] for row in out[4:8]] == [
    {"action": "open_url", "value": "https://example.org/golem"}, {"action": "run_command", "value": "/golem source"},
    {"action": "suggest_command", "value": "/msg "}, {"action": "copy_to_clipboard", "value": "4000 128 4000"}], out[4:8]
assert out[8]["json"] == ('{"text":"hover","hoverEvent":{"action":"show_text","contents":{"text":"the text of the '
                          'hover","color":"gray"}},"insertion":"Alex"}'), out[8]
assert out[8]["depth"] == 2 and out[8]["plain"] == "hover", out[8]
assert out[9]["json"] == '{"translate":"death.attack.generic","with":[{"text":"Alex"}],"color":"red"}', out[9]
assert out[9]["plain"] == "death.attack.generic[Alex]", out[9]
assert out[10]["parsed"]["text"] == "\u00e4\u00f6\u00fc \u2603 \U0001f600" and out[10]["utf16"] == len(out[10]["json"]) + 1, out[10]
assert out[11] == {"stripped": "a"} and out[12] == {"stripped": "plain words stay"}, out[11:]
assert log == [], log
for bad in ({"plain": "x", "color": "#ffaa0"}, {"plain": "x", "color": "orange"}, {"plain": "x", "color": "#gg0000"}):
    result = run([DEVTOOL, "text-json", write("bad-color.json", [bad])], cwd=P.root)
    assert result.returncode == 1 and "no colour" in result.stderr, (bad, result.stderr)

`Text::player` and `strip_codes` also drop the direction controls of Unicode, the nine characters
U+202A to U+202E and U+2066 to U+2069, each of them, and keep the characters beside them.

In [ ]:
CONTROLS = [chr(point) for point in (*range(0x202a, 0x202f), *range(0x2066, 0x206a))]
BESIDE = ["\u2029", "\u202f", "\u2065", "\u206a"]
rows, log = devtool("text-json", write("direction.json", [
    {"player": "Al" + "".join(CONTROLS) + "ex"},
    *({"strip": "a" + control + "b"} for control in CONTROLS),
    *({"strip": "a" + kept + "b"} for kept in BESIDE),
]))
out = rows["texts"]
print(len(CONTROLS), "controls:", out[0], [row["stripped"] for row in out[1:10]])
assert len(CONTROLS) == 9, CONTROLS
assert out[0]["json"] == '{"text":"Alex"}', "the text of a player keeps a direction control: %s" % out[0]
assert [row["stripped"] for row in out[1:10]] == ["ab"] * 9, out[1:10]
assert [row["stripped"] for row in out[10:]] == ["a" + kept + "b" for kept in BESIDE], out[10:]
assert log == [], log

The plugin mover sends a styled greeting through the helpers of the context, whose name of a player
keeps its text literal, to one player and to every player but one, and the context refuses a text
nested deeper than 32 and one whose JSON is longer than the game sends a client, each with a line
on the log. The wire form carries the UUID or the hidden players joined by commas, and the JSON as
one field.

In [ ]:
steps, log = script("styled", {
    "budgetMs": 200,
    "plugins": [{"kind": "mover", "name": "mover", "commands": ["move"], "to": NETHER}],
    "steps": [
        {"op": "command", "name": "move", "arguments": "text " + ALEX["uuid"] + " \u00a7kSam\n"},
        {"op": "command", "name": "move", "arguments": "shout " + SAM["uuid"] + " Alex"},
        {"op": "command", "name": "move", "arguments": "deep " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "long " + ALEX["uuid"]},
        {"op": "drain"},
    ],
})
assert [step["answer"] for step in steps[:4]] == [["sent true"], ["sent true"], ["sent false"], ["sent false"]], steps
greeting = ('{"text":"Hello ","color":"gold","extra":[{"text":"%s","color":"#ff5555","bold":true,"clickEvent":'
            '{"action":"open_url","value":"https://example.org/golem"},"hoverEvent":{"action":"show_text","contents":'
            '{"text":"a player","italic":true}}},{"text":"!"}]}')
assert steps[4]["decoded"] == [
    {"kind": "styledMessage", "uuid": ALEX["uuid"], "json": greeting % "Sam"},
    {"kind": "styledBroadcast", "hidden": [SAM["uuid"]], "json": greeting % "Alex"},
], steps[4]["decoded"]
assert steps[4]["answer"][1] == US.join(["styledBroadcast", SAM["uuid"], greeting % "Alex"]), steps[4]["answer"][1]
print(log)
assert log == ["a plugin asked to send a text 33 deep, deeper than 32, so the runtime refuses it",
               "a plugin asked to send a text whose JSON holds 262155 UTF-16 units, more than the 262144 that the game "
               "sends a client, so the runtime refuses it"], log
rows, _ = devtool("actions-encode", write("styled-actions.json", [
    {"kind": "styledBroadcast", "hidden": [ALEX["uuid"], SAM["uuid"]], "json": '{"text":"x"}'},
    {"kind": "styledBroadcast", "json": '{"text":"x"}'},
    {"wire": US.join(["styledMessage", ALEX["uuid"]])},
]))
assert rows[0]["fields"] == ["styledBroadcast", ALEX["uuid"] + "," + SAM["uuid"], '{"text":"x"}'], rows[0]
assert rows[0]["reencoded"] == rows[0]["wire"] and rows[1]["fields"][1] == "" and rows[1]["decoded"]["hidden"] == [], rows
assert rows[2]["decoded"] is None, rows[2]

## The game

A plugin asks the game through its context while a native of the server thread runs. The script
hands the natives a scripted game in place of the one the mod connects. The server thread is the
thread of the first `serverStarting`, so a question before it answers none, and the first such
question of the process is logged once. Then every question answers what the game holds, none
where the game holds none, and nothing for a player who is not online or a dimension the server
does not have: the respawn point with its angle and whether `/spawnpoint` forced it and where
the player would respawn, at the world spawn or in a chunk that is not loaded. A player whose
game mode the game does not have is left out of the players alone, as the natives leave out a
line that does not read, and a UUID reads in any case. A safe place keeps the dimension and the
rotation of its target and is found within the radius alone, held to 8 blocks, a target that is
no number gets no answer, and a search that meets a chunk that is not loaded answers so instead
of a place. The scripted game answers as the natives do, so the cells of the tools below judge
the core plugin and not the copy of a rule. The plugin ask asks for the players on every chat
message and every tick. A chat message that reaches the plugins on a thread of its own, as Forge
calls `chat` on a worker thread, gets no answer and adds no second line to the log, while a tick
and a chat message on the server thread, the thread of the first `serverStarting`, get theirs.
Without a game every question answers nothing.

In [ ]:
OVERWORLD = {"id": "minecraft:overworld", "spawn": {"x": 8, "y": 70, "z": -16, "angle": 90.0}, "minY": -64,
             "maxY": 320, "safe": {"x": 0.5, "y": 65.0, "z": 2.5}}
NETHER_DIMENSION = {"id": "minecraft:the_nether", "spawn": {"x": 8, "y": 70, "z": -16}, "minY": 0, "maxY": 256}
END_DIMENSION = {"id": "minecraft:the_end", "spawn": {"x": 100, "y": 50, "z": 0}, "minY": 0, "maxY": 256,
                 "safe": "unloaded"}
GAME = {
    "players": [
        {**ALEX, "displayName": "Alex the Brave", "ping": 12, "gameMode": "creative", "level": 2, "idleMs": 1500,
         "location": NETHER, "respawn": {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20, "angle": 45.0,
                                         "forced": True},
         "respawnPlace": {"dimension": "minecraft:overworld", "x": 10.5, "y": 64.1, "z": -19.5, "yaw": 45.0},
         "mainHand": {"id": "minecraft:diamond", "count": 3}},
        {**SAM, "listName": "[AFK] Sam", "idleMs": 900000, "location": LAVA, "respawnPlace": "unloaded"},
        {"uuid": "0f0e0d0c-0b0a-4908-8706-050403020100", "name": "Broken", "gameMode": "hardcore", "location": LAVA},
    ],
    "dimensions": [OVERWORLD, NETHER_DIMENSION, END_DIMENSION],
}
ASK = {"kind": "ask", "name": "ask", "commands": ["ask"], "on": ["chat", "tick"]}
questions = ["players", "player alex", "player " + SAM["uuid"].upper(), "player nobody", "location " + ALEX["uuid"],
             "respawn " + ALEX["uuid"], "respawn " + SAM["uuid"], "spawn minecraft:overworld", "spawn golem:void",
             "hand " + ALEX["uuid"], "hand " + SAM["uuid"], "heights minecraft:the_nether",
             "safe minecraft:overworld 0.2 64 2.9 3", "safe minecraft:the_nether 0 64 0 3", "safe golem:void 0 0 0 3",
             "safe minecraft:the_end 0 64 0 3", "location 00000000-0000-4000-8000-000000000000",
             "location " + ALEX["uuid"].upper(), "safe minecraft:overworld 0.2 64 2.9 0",
             "safe minecraft:overworld 0.2 57 2.9 50", "safe minecraft:overworld 0.2 56 2.9 50",
             "safe minecraft:overworld NaN 64 2.9 3", "respawnplace " + ALEX["uuid"], "respawnplace " + SAM["uuid"],
             "respawnplace nobody"]
steps, log = script("game", {
    "budgetMs": budget_ms,
    "plugins": [ASK],
    "game": GAME,
    "steps": [{"op": "command", "name": "ask", "arguments": "players"},
              {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"}]
             + [{"op": "command", "name": "ask", "arguments": question} for question in questions]
             + [{"op": "together", "steps": [{"op": "chat", **SAM, "message": "anyone here"}]},
                {"op": "tick", "nanos": 1000000},
                {"op": "chat", **ALEX, "message": "on the server thread"},
                {"op": "drain"}],
})
before = json.loads(steps[0]["answer"][0])
answers = {question: json.loads(step["answer"][0]) for question, step in zip(questions, steps[2:])}
for question, value in answers.items():
    print("%-60s %s" % (question[:60], json.dumps(value)[:120]))
assert before is None, before
players = answers["players"]
assert [player["name"] for player in players] == ["Alex", "Sam"], players
assert players[0] == {"uuid": ALEX["uuid"], "name": "Alex", "displayName": "Alex the Brave", "listName": None,
                      "ping": 12, "gameMode": "creative", "level": 2, "idleMs": 1500}, players[0]
assert players[1]["listName"] == "[AFK] Sam" and players[1]["gameMode"] == "survival" and players[1]["level"] == 0
assert answers["player alex"] == players[0] and answers["player " + SAM["uuid"].upper()] == players[1], answers
assert answers["player nobody"] == {"none": True}, answers["player nobody"]
assert answers["location " + ALEX["uuid"]] == NETHER, answers
assert answers["respawn " + ALEX["uuid"]] == {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20,
                                              "angle": 45.0, "forced": True}, answers["respawn " + ALEX["uuid"]]
assert answers["respawnplace " + ALEX["uuid"]] == {"dimension": "minecraft:overworld", "x": 10.5, "y": 64.1, "z": -19.5,
                                                   "yaw": 45.0, "pitch": 0.0}
assert answers["respawnplace " + SAM["uuid"]] == {"unloaded": True} and answers["respawnplace nobody"] is None
assert answers["respawn " + SAM["uuid"]] == {"none": True}
assert answers["spawn minecraft:overworld"] == {"dimension": "minecraft:overworld", "x": 8, "y": 70, "z": -16,
                                                 "angle": 90.0}
assert answers["spawn golem:void"] is None
assert answers["hand " + ALEX["uuid"]] == {"id": "minecraft:diamond", "count": 3}
assert answers["hand " + SAM["uuid"]] == {"none": True}
assert answers["heights minecraft:the_nether"] == {"min": 0, "max": 256}
assert answers["safe minecraft:overworld 0.2 64 2.9 3"] == {"dimension": "minecraft:overworld", "x": 0.5, "y": 65.0,
                                                           "z": 2.5, "yaw": 0.0, "pitch": 0.0}
assert answers["safe minecraft:the_nether 0 64 0 3"] == {"none": True}
assert answers["safe golem:void 0 0 0 3"] is None
assert answers["safe minecraft:the_end 0 64 0 3"] == {"unloaded": True}, answers["safe minecraft:the_end 0 64 0 3"]
assert answers["location 00000000-0000-4000-8000-000000000000"] is None
assert answers["location " + ALEX["uuid"].upper()] == NETHER, answers
assert answers["safe minecraft:overworld 0.2 64 2.9 0"] == {"none": True}
assert answers["safe minecraft:overworld 0.2 57 2.9 50"] == answers["safe minecraft:overworld 0.2 64 2.9 3"]
assert answers["safe minecraft:overworld 0.2 56 2.9 50"] == {"none": True}
assert answers["safe minecraft:overworld NaN 64 2.9 3"] is None
chat_off_thread = steps[-4]["answer"][0]
assert chat_off_thread["answer"] is False, chat_off_thread
assert broadcasts(steps[-1]) == ["ask got no answer", "ask saw 2 players", "ask saw 2 players"], steps[-1]
for line in log:
    print(line)
assert log == ["a plugin asked the game for players off the server thread, so the game answers none, and later "
               "questions like it are not logged"], log

without, log = script("no-game", {
    "budgetMs": budget_ms,
    "plugins": [ASK],
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {"op": "command", "name": "ask", "arguments": "players"},
              {"op": "command", "name": "ask", "arguments": "safe minecraft:overworld 0 64 0 3"}],
})
assert [json.loads(step["answer"][0]) for step in without[1:]] == [None, None], without
assert log == [], log

## The wire form of the game

The mod answers every question as fields joined by U+001F, with the numbers as Java writes them.
`game-wire` reads answers as the natives read them. A number in the exponent form of Java reads,
a coordinate that is no finite number, a game mode the game does not have, a boolean that is
neither `true` nor `false` and a wrong number of fields do not read, an operator level above 4
reads as 4 and one below 0 as 0, with one line on the log for the first of them, since the game
takes the level of `ops.json` without a bound, a ping below 0 counts as 0, an empty name in the
list is no name, and the empty answer of a respawn point, a hand, a place to respawn and a safe
place is none. A safe place keeps the dimension and the rotation of its target, a place to
respawn has the pitch 0, and the answer `unloaded` says that the search met a chunk that is not
loaded, or that a chunk around the respawn point is not loaded.

In [ ]:
def joined(*fields):
    return US.join(str(field) for field in fields)


PLAYER = [ALEX["uuid"], "Alex", "Alex", 12, "survival", 0, 1500, ""]
TARGET = {"dimension": "minecraft:the_nether", "x": 0.0, "y": 100.0, "z": 0.0, "yaw": 45.0, "pitch": 10.0}
entries = [
    {"question": "players", "answer": joined(*PLAYER)},
    {"question": "players", "answer": joined(*PLAYER[:7], "[AFK] Alex")},
    {"question": "players", "answer": joined(*PLAYER[:3], -3, *PLAYER[4:])},
    {"question": "players", "answer": joined(*PLAYER[:4], "hardcore", *PLAYER[5:])},
    {"question": "players", "answer": joined(*PLAYER[:5], 5, *PLAYER[6:])},
    {"question": "players", "answer": joined(*PLAYER[:7])},
    {"question": "location", "answer": joined("minecraft:overworld", 1.5, 64.0, -3.25, 90.0, -12.5)},
    {"question": "location", "answer": joined("minecraft:overworld", "1.0E-5", "-0.0", "3.0E10", "0.0", "0.0")},
    {"question": "location", "answer": joined("minecraft:overworld", "NaN", 64.0, 0.0, 0.0, 0.0)},
    {"question": "location", "answer": joined("minecraft:overworld", "Infinity", 64.0, 0.0, 0.0, 0.0)},
    {"question": "location", "answer": joined("minecraft:overworld", 1.5, 64.0, -3.25, 90.0)},
    {"question": "respawn", "answer": ""},
    {"question": "respawn", "answer": joined("minecraft:overworld", 10, 64, -20, "90.0", "true")},
    {"question": "respawn", "answer": joined("minecraft:overworld", "ten", 64, -20, "90.0", "true")},
    {"question": "spawn", "answer": joined(8, 70, -16, "90.0"), "dimension": "minecraft:overworld"},
    {"question": "mainHand", "answer": ""},
    {"question": "mainHand", "answer": joined("minecraft:diamond", 3)},
    {"question": "heights", "answer": joined(-64, 320)},
    {"question": "safeLocation", "answer": "", "target": TARGET},
    {"question": "safeLocation", "answer": joined("-0.5", "100.0", "0.5"), "target": TARGET},
    {"question": "safeLocation", "answer": "unloaded", "target": TARGET},
    {"question": "safeLocation", "answer": "loaded", "target": TARGET},
    {"question": "players", "answer": joined(*PLAYER[:5], -1, *PLAYER[6:])},
    {"question": "respawn", "answer": joined("minecraft:overworld", 10, 64, -20)},
    {"question": "respawn", "answer": joined("minecraft:overworld", 10, 64, -20, "0.0", "yes")},
    {"question": "respawnPlace", "answer": joined("minecraft:overworld", "10.5", "64.1", "-19.5", "-90.0")},
    {"question": "respawnPlace", "answer": ""},
    {"question": "respawnPlace", "answer": "unloaded"},
    {"question": "respawnPlace", "answer": joined("minecraft:overworld", "10.5", "64.1", "-19.5")},
]
rows, log = devtool("game-wire", write("game-wire.json", entries))
decoded = [row["decoded"] for row in rows]
for row in rows:
    print("%-12s %-60r %s" % (row["question"], row["answer"].replace(US, "|")[:60], json.dumps(row["decoded"])[:80]))
expected_player = {"uuid": ALEX["uuid"], "name": "Alex", "displayName": "Alex", "listName": None, "ping": 12,
                   "gameMode": "survival", "level": 0, "idleMs": 1500}
assert decoded[0] == expected_player, decoded[0]
assert decoded[1] == {**expected_player, "listName": "[AFK] Alex"}, decoded[1]
assert decoded[2] == {**expected_player, "ping": 0}, decoded[2]
assert decoded[3] is None and decoded[5] is None, decoded[3:6]
assert decoded[4] == {**expected_player, "level": 4} and decoded[22] == expected_player, (decoded[4], decoded[22])
assert decoded[6] == {"dimension": "minecraft:overworld", "x": 1.5, "y": 64.0, "z": -3.25, "yaw": 90.0,
                      "pitch": -12.5}, decoded[6]
assert decoded[7] == {"dimension": "minecraft:overworld", "x": 1e-05, "y": -0.0, "z": 3e10, "yaw": 0.0,
                      "pitch": 0.0}, decoded[7]
assert decoded[8:11] == [None, None, None], decoded[8:11]
assert decoded[11] == {"none": True} and decoded[13] is None, decoded[11:14]
assert decoded[12] == {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20, "angle": 90.0,
                       "forced": True}, decoded[12]
assert decoded[23] is None and decoded[24] is None, "a respawn point without its angle or with a word reads"
assert decoded[25] == {"dimension": "minecraft:overworld", "x": 10.5, "y": 64.1, "z": -19.5, "yaw": -90.0,
                       "pitch": 0.0}, decoded[25]
assert decoded[26] == {"none": True} and decoded[27] == {"unloaded": True} and decoded[28] is None, decoded[26:29]
assert decoded[14] == {"dimension": "minecraft:overworld", "x": 8, "y": 70, "z": -16, "angle": 90.0}, decoded[14]
assert decoded[15] == {"none": True} and decoded[16] == {"id": "minecraft:diamond", "count": 3}, decoded[15:17]
assert decoded[17] == {"min": -64, "max": 320}, decoded[17]
assert decoded[18] == {"none": True}, decoded[18]
assert decoded[19] == {**TARGET, "x": -0.5, "y": 100.0, "z": 0.5}, decoded[19]
assert decoded[20] == {"unloaded": True} and decoded[21] is None, decoded[20:22]
assert log == ["the game answered the operator level 5 for Alex, which lies outside 0 to 4, so Golem reads it as 4, "
               "and later levels like it are not logged"], log

## The tools of operators

`/golem whoami` tells every sender how Golem sees them. `/golem player`, `world`, `teleport` and
`listname` ask the game, and only the console, the operators of level 2 and more and a player who
holds the node of the tool, such as `golem.player`, may run them. A rule of the permissions decides
first. Alex holds `golem.*` through the group `admin` of the permissions of the script, which deny
`golem.listname`, so Alex runs the other tools also at a source of level 0, while `whoami` names
both levels, and is refused `listname`, although Alex is an operator of level 2. Where no rule
names the node, the operator level decides, as it did before the permissions, so no operator loses
a tool to them: Sam, of level 0, and a source of level 1 are refused, and with permissions that
leave Alex out of the group, and in a script without permissions, Alex keeps every tool at a source
of level 2 and is refused at a source of level 0, since a player counts at the lower of their
operator level and the level of the source. When the game gives no answer for the players, a player
whose node no rule names reads that the game gives no answer, not that they are no operator.
`player` names where a player is and how they play, and `world` the
spawn and the heights of a dimension. `teleport` queues a teleport to the nearest safe place
within 3 blocks, `SAFE_RADIUS`, with the rotation the player has and says that it queued it, since
the mod runs it at the end of the tick, or says that there is none or that its chunks are not
loaded, and refuses a target outside the bounds of the world that `tp` takes. `listname` queues
the name of the player list as it is typed, runs of spaces included, or its clearing. Unknown
players and dimensions and words that are no numbers answer what is wrong.

In [ ]:
CONSOLE = {"op": "command", "name": "golem"}
AS_ALEX = {**CONSOLE, "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 2}
AS_SAM = {**CONSOLE, "senderUuid": SAM["uuid"], "senderName": SAM["name"], "level": 0}
AS_ALEX_LOW = {**AS_ALEX, "level": 0}
BLOCK = {**CONSOLE, "senderName": "@", "level": 2}
LOW = {**CONSOLE, "senderName": "Server", "level": 1}
runs = [
    (CONSOLE, "whoami"), (AS_ALEX, "whoami"), (AS_SAM, "whoami"), (BLOCK, "whoami"),
    (AS_ALEX_LOW, "WHOAMI"), (AS_ALEX_LOW, "player SAM"),
    (CONSOLE, "player Alex"), (AS_ALEX, "player sam"), (AS_SAM, "player Alex"), (BLOCK, "player Alex"),
    (LOW, "player Alex"), (CONSOLE, "player Nobody"), (CONSOLE, "player"),
    (CONSOLE, "world"), (CONSOLE, "world minecraft:the_nether"), (CONSOLE, "world golem:void"),
    (CONSOLE, "teleport Alex minecraft:overworld 0.2 64 2.9"), (CONSOLE, "teleport Alex minecraft:overworld 0.2 61 2.9"),
    (CONSOLE, "teleport Alex minecraft:overworld 0.2 62 2.9"), (CONSOLE, "teleport Sam minecraft:the_nether 0 64 0"),
    (CONSOLE, "teleport Alex golem:void 0 64 0"), (CONSOLE, "teleport Alex minecraft:overworld x 64 0"),
    (CONSOLE, "teleport Alex minecraft:the_end 0 64 0"), (CONSOLE, "teleport Alex minecraft:overworld 30000001 64 0"),
    (CONSOLE, "teleport Alex minecraft:overworld 0 -20000001 0"),
    (AS_SAM, "teleport Sam minecraft:overworld 0 64 0"),
    (CONSOLE, "listname Alex [AFK] Alex"), (CONSOLE, "listname Alex"), (AS_SAM, "listname Sam x"),
    (CONSOLE, "listname Alex [AFK]   Alex"), (CONSOLE, "listname Nobody x"),
    (CONSOLE, "PLAYER Alex"), (CONSOLE, "player Broken"), (AS_ALEX, "listname Alex x"),
]
steps, log = script("operators", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "core"}],
    "permissions": PERMISSIONS,
    "game": GAME,
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"}]
             + [{**sender, "arguments": arguments} for sender, arguments in runs]
             + [{"op": "drain"}],
})
said = {(sender.get("senderName", "console"), arguments): step["answer"] for (sender, arguments), step
        in zip(runs, steps[1:])}
for key, lines in said.items():
    print("%-8s %-45s %s" % (key[0], key[1][:45], " / ".join(lines)[:160]))
refused = ["Only the console, operators of level 2 and more and a player who holds golem.%s may run /golem %s." % (
    name, name) for name in ("player", "teleport", "listname")]
assert said[("console", "whoami")] == ["You run /golem as the console."]
assert said[("Alex", "whoami")] == ["You run /golem as the player Alex at the operator level 2."]
assert said[("Sam", "whoami")] == ["You run /golem as the player Sam at the operator level 0."]
assert said[("@", "whoami")] == ["You run /golem as @ at the level 2."]
assert said[("Alex", "WHOAMI")] == ["You run /golem as the player Alex at the operator level 2, from a source at the "
                                    "level 0."]
assert said[("Alex", "player SAM")][0].startswith("Sam ("), said[("Alex", "player SAM")]
assert said[("Alex", "listname Alex x")] == [refused[2]], said[("Alex", "listname Alex x")]
alex = said[("console", "player Alex")]
assert alex == [
    "Alex (%s): minecraft:the_nether 1.50 64.00 -2.25, yaw 90.0, pitch -12.5" % ALEX["uuid"],
    "Game mode creative, ping 12 ms, operator level 2, idle 1.5 s",
    "Display name: Alex the Brave",
    "List name: none",
    "Respawn point: minecraft:overworld 10 64 -20",
    "Main hand: 3 minecraft:diamond",
], alex
assert said[("Alex", "player sam")][3:] == ["List name: [AFK] Sam", "Respawn point: none", "Main hand: empty"]
assert said[("Sam", "player Alex")] == [refused[0]] and said[("Server", "player Alex")] == [refused[0]]
assert said[("@", "player Alex")] == alex and said[("console", "PLAYER Alex")] == alex
assert said[("console", "player Nobody")] == ["No player Nobody is online."]
assert said[("console", "player")] == ["Usage: /golem player <name>"]
assert said[("console", "world")] == ["minecraft:overworld: spawn 8 70 -16 at the angle 90.0, blocks from y -64 "
                                      "below y 320"]
assert said[("console", "world minecraft:the_nether")] == ["minecraft:the_nether: spawn 8 70 -16 at the angle "
                                                           "0.0, blocks from y 0 below y 256"]
assert said[("console", "world golem:void")] == ["The server has no dimension golem:void, or the game gives no "
                                                 "answer now."]
assert said[("console", "teleport Alex minecraft:overworld 0.2 64 2.9")] == [
    "Queued a teleport of Alex to minecraft:overworld 0.50 65.00 2.50, yaw 90.0, pitch -12.5."]
bounds = ["Usage: /golem teleport <name> <dimension> <x> <y> <z>, with x and z from -30000000 to below 30000000 and y "
          "from -20000000 to below 20000000"]
assert said[("console", "teleport Alex minecraft:overworld 30000001 64 0")] == bounds
assert said[("console", "teleport Alex minecraft:overworld 0 -20000001 0")] == bounds
assert said[("console", "teleport Alex minecraft:overworld 0.2 61 2.9")] == [
    "No safe place within 3 blocks of minecraft:overworld 0.20 61.00 2.90, yaw 90.0, pitch -12.5, so Alex stays."]
assert said[("console", "teleport Alex minecraft:overworld 0.2 62 2.9")] == said[
    ("console", "teleport Alex minecraft:overworld 0.2 64 2.9")], "the tool does not search within 3 blocks"
assert said[("console", "player Broken")] == ["No player Broken is online."]
assert said[("console", "teleport Sam minecraft:the_nether 0 64 0")] == [
    "No safe place within 3 blocks of minecraft:the_nether 0.00 64.00 0.00, yaw -45.0, pitch 0.0, so Sam stays."]
assert said[("console", "teleport Alex golem:void 0 64 0")] == ["The server has no dimension golem:void, or the "
                                                                "game gives no answer now."]
assert said[("console", "teleport Alex minecraft:overworld x 64 0")] == [
    "Usage: /golem teleport <name> <dimension> <x> <y> <z>, with numbers for x, y and z"]
assert said[("console", "teleport Alex minecraft:the_end 0 64 0")] == [
    "The chunks near minecraft:the_end 0.00 64.00 0.00, yaw 90.0, pitch -12.5 are not loaded, so Alex stays, since "
    "the search for a safe place loads no chunk."]
assert said[("Sam", "teleport Sam minecraft:overworld 0 64 0")] == [refused[1]]
assert said[("console", "listname Alex [AFK] Alex")] == ["The player list names Alex [AFK] Alex."]
assert said[("console", "listname Alex")] == ["The player list shows the name of Alex again."]
assert said[("Sam", "listname Sam x")] == [refused[2]]
assert said[("console", "listname Alex [AFK]   Alex")] == ["The player list names Alex [AFK]   Alex."]
assert said[("console", "listname Nobody x")] == ["No player Nobody is online."]
assert steps[-1]["decoded"] == [
    {"kind": "teleport", "id": 1, "uuid": ALEX["uuid"], "dimension": "minecraft:overworld", "x": 0.5, "y": 65.0,
     "z": 2.5, "yaw": 90.0, "pitch": -12.5, "safe": None},
    {"kind": "teleport", "id": 2, "uuid": ALEX["uuid"], "dimension": "minecraft:overworld", "x": 0.5, "y": 65.0,
     "z": 2.5, "yaw": 90.0, "pitch": -12.5, "safe": None},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK]   Alex"},
], steps[-1]
assert log == ["the bridge is connected: Forge 47.4.18, Minecraft 1.20.1"], log

# The permissions without Alex in the group admin, so no rule names a node of Golem for any player.
NO_MEMBER = PERMISSIONS.replace('[players.%s]\ngroups = ["admin"]\n' % ALEX["uuid"], "")
assert NO_MEMBER != PERMISSIONS and "players." not in NO_MEMBER, NO_MEMBER
by_level = [(AS_ALEX, "player sam"), (AS_ALEX_LOW, "player SAM"), (AS_SAM, "player Alex"), (AS_ALEX, "listname Alex x"),
            (BLOCK, "player Alex"), (LOW, "player Alex")]
for name, permissions in (("operators-by-level", {"permissions": NO_MEMBER}), ("operators-without-permissions", {})):
    levels, _ = script(name, {
        "plugins": [{"kind": "core"}],
        **permissions,
        "game": GAME,
        "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"}]
                 + [{**sender, "arguments": arguments} for sender, arguments in by_level],
    })
    answers = [step["answer"] for step in levels[1:]]
    assert answers[0] == said[("Alex", "player sam")], "%s: an operator of level 2 lost a tool: %s" % (name, answers[0])
    assert answers[1] == [refused[0]], "%s: a source of level 0 borrowed the level of Alex: %s" % (name, answers[1])
    assert answers[2] == [refused[0]], "%s: Sam, of level 0, ran a tool: %s" % (name, answers[2])
    assert answers[3] == ["The player list names Alex x."], "%s: %s" % (name, answers[3])
    assert answers[4] == alex and answers[5] == [refused[0]], "%s: %s" % (name, answers[4:])

silent, _ = script("operators-no-answer", {
    "plugins": [{"kind": "core"}],
    "permissions": NO_MEMBER,
    "game": {"players": None, "dimensions": [OVERWORLD]},
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {**AS_ALEX, "arguments": "player Alex"}, {**AS_ALEX, "arguments": "whoami"},
              {**CONSOLE, "arguments": "player Alex"}],
})
NO_ANSWER = ["The game gives no answer now, so try again."]
assert [step["answer"] for step in silent[1:]] == [
    NO_ANSWER, ["You run /golem as the player Alex from a source at the level 2."], NO_ANSWER], silent
# With a rule that grants the node, the tool asks the game for no level, so it answers its own line.
granted, _ = script("operators-no-answer-with-a-rule", {
    "plugins": [{"kind": "core"}],
    "permissions": PERMISSIONS,
    "game": {"players": None, "dimensions": [OVERWORLD]},
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {**AS_ALEX, "arguments": "world"}, {**AS_ALEX, "arguments": "listname Alex x"}],
})
assert granted[1]["answer"][0].startswith("minecraft:overworld: spawn "), granted[1]
assert granted[2]["answer"] == [refused[2]], granted[2]

## The plugin of the permissions and the PermissionAPI

The plugin `golem-permissions` of the runtime answers every Boolean node of the PermissionAPI of
Forge for an online player as LuckPerms answered it through Arclight: a node that a rule names as
the rule says, after it lowercased the node, and a node that nothing names true for an operator,
a player on the list of operators at a level of 1 or more, and false for every other player. A
player who is not online gets no answer, so the node falls to its default resolver, as Arclight
left an offline player to it. On the server thread the plugin asks the game for the level, and on
any other thread it answers from the levels it keeps from the joins, so a check of the thread of
a mod, such as the thread of Simple Voice Chat, answers Alex, an operator, as an operator until
Alex leaves.

In [ ]:
FORGE_NODES = ["voicechat.listen", "voicechat.speak", "VoiceChat.Listen"]
API_PERMISSIONS = '''
[groups.default]
nodes = ["voicechat.listen"]

[players.%s]
nodes = ["-voicechat.listen"]
''' % SAM["uuid"]
NOBODY = {"uuid": "00000000-0000-4000-8000-00000000abcd", "name": "Nobody"}
asks = [{"op": "permission", "uuid": who["uuid"], "node": node} for who in (ALEX, SAM, NOBODY) for node in FORGE_NODES]
LEVELS = {"players": [{**ALEX, "level": 4, "location": NETHER}, {**SAM, "level": 0, "location": LAVA}],
          "dimensions": [OVERWORLD]}
api, log = script("permission-api", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "permissions"}],
    "permissions": API_PERMISSIONS,
    "game": LEVELS,
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {"op": "playerJoined", **ALEX}, {"op": "playerJoined", **SAM}]
             + asks
             + [{"op": "together", "steps": [{"op": "permission", "uuid": ALEX["uuid"], "node": "voicechat.speak"},
                                             {"op": "permission", "uuid": NOBODY["uuid"], "node": "voicechat.speak"}]},
                {"op": "playerLeft", **ALEX},
                {"op": "together", "steps": [{"op": "permission", "uuid": ALEX["uuid"], "node": "voicechat.speak"}]}],
})
answers = [step["answer"] for step in api[3:3 + len(asks)]]
print("on the server thread:", answers)
assert answers == [True, True, True, False, False, False, None, None, None], answers
beside = [step["answer"] for step in api[3 + len(asks)]["answer"]]
after = [step["answer"] for step in api[-1]["answer"]]
print("on another thread:", beside, "after Alex left:", after)
assert beside == [True, None] and after == [None], (beside, after)
assert log == [], log

## The operator levels that the plugin keeps

A check of another thread answers from the operator levels that the plugin keeps, since it asks
the game on the server thread alone. The plugin takes the level of a player from the game at their
join, reads the levels of the players online again on every twentieth tick, which is once a second,
and forgets a player at their leave and every player at the stop of the server. The cell holds each
of these steps with a node that nothing names, which an operator holds and no other player. Alex,
an operator of the lowest level, 1, holds it on the server thread and beside it. Olga, an operator
whom the game names and whose join the plugin never saw, gets no answer beside the server thread
until the twentieth tick reads the players. Nobody, whose join the plugin saw and whom the game
does not name, counts as a player without a level until that read drops them. After Sam left, and
after `serverStopping` for everybody, a check beside the server thread gets no answer.

In [ ]:
OLGA = {"uuid": "01a0c0de-0000-4000-8000-0000000000aa", "name": "Olga"}
SPEAK = "voicechat.speak"
KEPT_LEVELS = {"players": [{**ALEX, "level": 1, "location": NETHER}, {**SAM, "level": 0, "location": LAVA},
                           {**OLGA, "level": 2, "location": NETHER}],
               "dimensions": [OVERWORLD]}


def beside_thread(*players):
    # One check of SPEAK for each of the players, each on a thread of its own.
    return {"op": "together", "steps": [{"op": "permission", "uuid": who["uuid"], "node": SPEAK} for who in players]}


def together(step):
    return [call["answer"] for call in step["answer"]]


kept, log = script("permission-levels", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "permissions"}],
    "permissions": API_PERMISSIONS,
    "game": KEPT_LEVELS,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "playerJoined", **ALEX}, {"op": "playerJoined", **SAM}, {"op": "playerJoined", **NOBODY},
        {"op": "permission", "uuid": ALEX["uuid"], "node": SPEAK},
        {"op": "permission", "uuid": SAM["uuid"], "node": SPEAK},
        beside_thread(ALEX, SAM, NOBODY, OLGA),
        {"op": "tick", "nanos": 1000000, "repeat": 19},
        beside_thread(NOBODY, OLGA),
        {"op": "tick", "nanos": 1000000},
        beside_thread(ALEX, SAM, NOBODY, OLGA),
        {"op": "playerLeft", **SAM},
        beside_thread(SAM),
        {"op": "serverStopping"},
        beside_thread(ALEX, OLGA),
    ],
})
print("on the server thread:", kept[4]["answer"], kept[5]["answer"])
for label, index in (("after the joins", 6), ("after 19 ticks", 8), ("after 20 ticks", 10), ("after Sam left", 12),
                     ("after serverStopping", 14)):
    print("beside it %s: %s" % (label, together(kept[index])))
assert [kept[4]["answer"], kept[5]["answer"]] == [True, False], "an operator of level 1 is no operator: %s" % kept[4:6]
assert together(kept[6]) == [True, False, False, None], kept[6]
assert together(kept[8]) == [False, None], "the plugin read the players before its twentieth tick: %s" % kept[8]
assert together(kept[10]) == [True, False, None, True], "the twentieth tick did not read the players: %s" % kept[10]
assert together(kept[12]) == [None], kept[12]
assert together(kept[14]) == [None, None], "serverStopping left a level behind: %s" % kept[14]
assert log == [], log

## The commands of LuckPerms in the runtime

The plugin of the permissions owns `/luckperms` and its aliases. A command of the console writes
its change to the store and installs it at once, so the next check of the same tick sees it: Alex
holds `golem.player` once the console gave them the group `admin`, and the tool of the core plugin
answers them. A player who holds no node of the commands is refused. The store holds the change,
and the resolver holds it under the generation of before the write, so the reload of the next tick
finds the same set and draws level. A change that another
writer puts into the store behind the resolver reaches it once a tick of the plugin wakes its
reload, and the store closes after the last step with every change in its file.

In [ ]:
LP_STORE = SCRATCH / "lp.db"
LP = {"op": "command", "name": "lp"}
lp_steps, lp_log = script("luckperms", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "core"}, {"kind": "permissions"}, {"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS.replace('[players.%s]\ngroups = ["admin"]\n' % ALEX["uuid"], ""),
    "store": str(LP_STORE),
    "game": GAME,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "commandNames"},
        {**CHECK, **SENDERS[1][1], "arguments": "golem.player"},
        {**LP, "arguments": "user Alex parent add admin"},
        {"op": "generation"},
        {**CHECK, **SENDERS[1][1], "arguments": "golem.player"},
        {"op": "command", "name": "golem", **SENDERS[1][1], "arguments": "player Sam"},
        {**LP, **SENDERS[2][1], "arguments": "user Sam parent add admin"},
        {"op": "command", "name": "perm", "arguments": "user Sam permission set golem.player"},
        {"op": "stored"},
        {"op": "behind", "group": "default", "nodes": ["essentials.home", "essentials.fly"]},
        {"op": "tick", "nanos": 1000000},
        {"op": "sleep", "ms": 300},
        {"op": "generation"},
        {**CHECK, **SENDERS[2][1], "arguments": "essentials.fly"},
    ],
})
lp_answers = [step["answer"] for step in lp_steps]
for answer in lp_answers:
    print(answer)
assert lp_answers[1] == ["golem", "luckperms", "lp", "perm", "perms", "permission", "permissions", "check"], lp_answers[1]
assert lp_answers[2] == ["golem.player: granted false, set false, has false, No rule names golem.player"], lp_answers[2]
assert lp_answers[3] == ["[LP] Alex now inherits permissions from admin in context global."], lp_answers[3]
assert lp_answers[4] == {"resolver": 1, "store": 2}, "the resolver did not take the change at once: %s" % lp_answers[4]
assert lp_answers[5] == ["golem.player: granted true, set true, has true, Group admin grants golem.*"], lp_answers[5]
assert lp_answers[6][0].startswith("Sam (%s): " % SAM["uuid"]), lp_answers[6]
assert lp_answers[7] == ["[LP] Running LuckPerms v5.5.71, as ported to Golem."], lp_answers[7]
assert lp_answers[8] == ["[LP] Set golem.player to true for Sam in context global."], lp_answers[8]
assert '[players.%s]\ngroups = ["admin"]' % ALEX["uuid"] in lp_answers[9], lp_answers[9]
assert '[players.%s]\nnodes = ["golem.player"]' % SAM["uuid"] in lp_answers[9], lp_answers[9]
assert lp_answers[10] is True and lp_answers[13] == {"resolver": 4, "store": 4}, lp_answers[10:14]
assert lp_answers[14] == ["essentials.fly: granted true, set true, has true, Group default grants essentials.fly"]
# The store closed after the last step, so a second open reads from the file what the commands wrote.
reopen = write("reopen.json", {"steps": [{"op": "open", "path": str(LP_STORE)},
                                         {"op": "player_permissions", "uuid": SAM["uuid"]}, {"op": "close"}]})
done = run([DEVTOOL, "store", reopen], cwd=P.root, timeout=300)
assert done.returncode == 0, done.stderr
reopened = [json.loads(line) for line in done.stdout.splitlines()]
print("Sam in the file:", reopened[1]["ok"])
assert reopened[1]["ok"] == {"uuid": SAM["uuid"], "groups": [], "nodes": ["golem.player"]}, reopened[1]
assert lp_log == ["the bridge is connected: Forge 47.4.18, Minecraft 1.20.1"], lp_log

## A command on a set that does not load

A command builds on the set of the store. When another writer left a set in the store that
`golem-permissions` refuses, such as one whose inheritance is a cycle, the resolver keeps the older
set for the checks, and a command that built on that older set would write it back over the store.
So the plugin runs no command then: a command that changes the permissions and one that only reads
them both answer that the set does not load, the store keeps what it holds, and the checks go on
with the set of the resolver. The reason can name a group, a node or the UUID of a player of the
configuration, so the console reads it, and so does Alex, who holds a node of the commands by the
set that the resolver keeps. Sam holds none and reads the line without the reason.

In [ ]:
UNLOADED_STORE = SCRATCH / "unloaded.db"
unloaded, unloaded_log = script("unloaded", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "permissions"}, {"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS.replace('[players.%s]\ngroups = ["admin"]\n' % ALEX["uuid"], ""),
    "store": str(UNLOADED_STORE),
    "game": GAME,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {**LP, "arguments": "user Alex parent add admin"},
        {**LP, "arguments": "user Alex permission set luckperms.user.info"},
        {"op": "behind", "group": "default", "inherits": ["admin"], "nodes": ["essentials.home"]},
        {**LP, "arguments": "user Sam parent add admin"},
        {**LP, "arguments": "user Alex info"},
        {**LP, **SENDERS[1][1], "arguments": "user Alex info"},
        {**LP, **SENDERS[2][1], "arguments": "user Alex info"},
        {"op": "generation"},
        {"op": "stored"},
        {**CHECK, **SENDERS[1][1], "arguments": "golem.player"},
    ],
})
unloaded_answers = [step["answer"] for step in unloaded]
for answer in unloaded_answers:
    print(answer)
NOT_LOADED = ["[LP] The permissions of Golem do not load, so no command runs: groups.default.inherits: the "
              "inheritance admin, default, admin is a cycle."]
assert unloaded_answers[1] == ["[LP] Alex now inherits permissions from admin in context global."], unloaded_answers[1]
assert unloaded_answers[2] == ["[LP] Set luckperms.user.info to true for Alex in context global."], unloaded_answers[2]
assert unloaded_answers[3] is True, unloaded_answers[3]
assert unloaded_answers[4] == NOT_LOADED and unloaded_answers[5] == NOT_LOADED, unloaded_answers[4:6]
# Alex holds a node of the commands by the set that the resolver keeps and reads the reason, as the console
# does. Sam holds none and reads the line alone, since the reason names groups of the configuration.
assert unloaded_answers[6] == NOT_LOADED, "a holder of a node of the commands reads no reason: %s" % (
    unloaded_answers[6])
assert unloaded_answers[7] == ["[LP] The permissions of Golem do not load, so no command runs."], (
    "a player without a node of the commands reads more than the line: %s" % unloaded_answers[7])
assert unloaded_answers[8] == {"resolver": 2, "store": 4}, unloaded_answers[8]
assert "[players.%s]" % SAM["uuid"] not in unloaded_answers[9], "a refused command wrote to the store: %s" % unloaded_answers[9]
assert 'inherits = ["admin"]' in unloaded_answers[9], "the store lost the set of the other writer: %s" % unloaded_answers[9]
assert unloaded_answers[10] == ["golem.player: granted true, set true, has true, Group admin grants golem.*"], (
    unloaded_answers[10])
# No tick woke the reload, so the log holds no line on the set yet.
assert unloaded_log == [], unloaded_log

## A change that the store refuses

The store takes a node and the name of a group of at most 256 bytes, and `golem-permissions` has no
such bound, so a command of the port can build a change that the store refuses. The plugin must
then answer the line of the command and the line of LuckPerms for a failed save, log why the store
refused, and go on with the set that the store holds: the check of the node must find no rule, the
generations of the store and of the resolver must stay where the last change that the store took
left them, and the store must hold that change alone. The cell refuses a node of a player, a node
of a group and a group, each by its own line of LuckPerms.

In [ ]:
LONG = "golem." + "x" * 294
WIDE = "g" * 257
NOT_TAKEN = "the store did not take a change of /lp %s: %s"
# The arguments of a command, its line, the line of its failed save and why the store refuses it.
REFUSALS = [
    ("user Sam permission set " + LONG, "[LP] Set %s to true for Sam in context global." % LONG,
     "[LP] There was an error whilst saving user data for Sam.",
     'a node of a player "%s" has 300 bytes, more than 256' % LONG),
    ("group default permission set " + LONG, "[LP] Set %s to true for default in context global." % LONG,
     "[LP] There was an error whilst saving group data for default.",
     'a node of a group "%s" has 300 bytes, more than 256' % LONG),
    ("creategroup " + WIDE, "[LP] %s was successfully created." % WIDE,
     "[LP] There was an error whilst creating %s." % WIDE,
     'the name of a group "%s" has 257 bytes, more than 256' % WIDE),
]
refused_steps, refused_log = script("luckperms-refused", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "permissions"}, {"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS,
    "store": str(SCRATCH / "refused.db"),
    "game": GAME,
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {**LP, "arguments": "user Sam permission set essentials.fly"}, {"op": "generation"}]
             + [{**LP, "arguments": arguments} for arguments, _, _, _ in REFUSALS]
             + [{"op": "generation"}, {**CHECK, **SENDERS[2][1], "arguments": "essentials.fly " + LONG},
                {"op": "stored"}],
})
taken, before, *refusals, after, checks, kept_set = [step["answer"] for step in refused_steps[1:]]
assert taken == ["[LP] Set essentials.fly to true for Sam in context global."], taken
assert before == {"resolver": 1, "store": 2}, before
for (arguments, line, save, _), answer in zip(REFUSALS, refusals, strict=True):
    print("/lp %s... answered %d lines, the last: %s" % (arguments[:36], len(answer), answer[-1][:72]))
    assert answer == [line, save], "/lp %s answered %s" % (arguments, answer)
assert after == {"resolver": 2, "store": 2}, "a change that the store refused moved a generation: %s" % after
assert checks == ["essentials.fly: granted true, set true, has true, The player grants essentials.fly",
                  "%s: granted false, set false, has false, No rule names %s" % (LONG, LONG)], checks
held = tomllib.loads(kept_set)
assert held["players"] == {ALEX["uuid"]: {"groups": ["admin"]}, SAM["uuid"]: {"nodes": ["essentials.fly"]}}, held
assert sorted(held["groups"]) == ["admin", "default"] and LONG not in kept_set, kept_set
assert refused_log == [NOT_TAKEN % (arguments, why) for arguments, _, _, why in REFUSALS], refused_log
print("the store refused %d changes, and the checks kept the set of the store" % len(REFUSALS))

## A command while the writer of the store is behind

The resolver of the permissions keeps the handle of the store that never waits and hands out no
other, so no command of the plugin waits for the writer of the store on the server thread. A
command asks the store for room for all of its changes before the first, so it makes all of them or
none. The cell holds the writer still and fills the store until one change more fits. `lp
deletegroup admin` makes two changes, its member and the group, so it must change nothing, answer
that the store takes no change now and leave the store, the resolver and the checks as they were. A
command of one change takes the room that is left, and the next one is refused as the first. The
plugin `keeper` then sets a setting through the store that the resolver hands out, which the store
must refuse at once, and no change may have reached the file while the writer stood still. Once the
runtime flushed the store on `ServerStopped`, the command runs.

In [ ]:
BUSY_STORE = SCRATCH / "busy.db"
NO_ROOM = ["[LP] The store of Golem takes no change now, so nothing changed. Run the command again later."]
crowded, crowded_log = script("luckperms-behind", {
    "budgetMs": 60000,
    "plugins": [{"kind": "keeper", "name": "keeper", "commands": ["keep"]}, {"kind": "permissions"},
                {"kind": "checker", "name": "checker", "commands": ["check"]}],
    "permissions": PERMISSIONS,
    "store": str(BUSY_STORE),
    "game": GAME,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {**KEEP, "arguments": "pause %d" % HELD_MS},
        {**KEEP, "arguments": "fill %d" % (BOUND - 1)},
        {**KEEP, "arguments": "stats"},
        {**LP, "arguments": "deletegroup admin"},
        {**KEEP, "arguments": "stats"},
        {"op": "generation"},
        {"op": "stored"},
        {**CHECK, **SENDERS[1][1], "arguments": "golem.player"},
        {**LP, "arguments": "user Sam permission set essentials.fly"},
        {**LP, "arguments": "user Sam permission set golem.player"},
        {**KEEP, "arguments": "resolver late x"},
        {**KEEP, "arguments": "stats"},
        {"op": "serverStopped"},
        {**LP, "arguments": "deletegroup admin"},
        {"op": "stored"},
    ],
})
crowded_answers = [step["answer"] for step in crowded]
for answer in crowded_answers:
    print(answer)
assert crowded_answers[1] == ["paused %d ms" % HELD_MS], crowded_answers[1]
assert crowded_answers[2] == ["took %d of %d" % (BOUND - 1, BOUND - 1)], crowded_answers[2]
filled = re.fullmatch(r"queued (\d+), written (\d+), pending (\d+)", crowded_answers[3][0])
assert filled and int(filled[3]) == BOUND - 1, "the plugin outran the pause: %s" % crowded_answers[3]
# Two changes do not fit, so the command makes neither: nothing joins the queue, and the store, the resolver and
# the checks hold the group as before.
assert crowded_answers[4] == NO_ROOM, "a command that the store has no room for answered %s" % crowded_answers[4]
assert crowded_answers[5] == crowded_answers[3], "a command without room changed the store: %s" % crowded_answers[5]
assert crowded_answers[6]["resolver"] == crowded_answers[6]["store"], crowded_answers[6]
crowded_set = crowded_answers[7]
assert '[players.%s]\ngroups = ["admin"]' % ALEX["uuid"] in crowded_set and "[groups.admin]" in crowded_set, (
    "a command without room changed the set of the store: %s" % crowded_set)
assert crowded_answers[8] == ["golem.player: granted true, set true, has true, Group admin grants golem.*"], (
    crowded_answers[8])
# One change fits, and the next one does not.
assert crowded_answers[9] == ["[LP] Set essentials.fly to true for Sam in context global."], crowded_answers[9]
assert crowded_answers[10] == NO_ROOM, crowded_answers[10]
# The store that the resolver hands out refuses a change at the bound and does not wait for the writer.
assert crowded_answers[11] == ["refused late: " + busy], (
    "the store of the resolver waited for the writer or took the change: %s" % crowded_answers[11])
assert crowded_answers[12] == ["queued %d, written %s, pending %d" % (int(filled[1]) + 1, filled[2], BOUND)], (
    "a change reached the file while the writer stood still: %s" % crowded_answers[12])
assert crowded_answers[14] == ["[LP] admin was successfully deleted."], crowded_answers[14]
assert "[groups.admin]" not in crowded_answers[15] and ALEX["uuid"] not in crowded_answers[15], crowded_answers[15]
assert '[players.%s]\nnodes = ["essentials.fly"]' % SAM["uuid"] in crowded_answers[15], crowded_answers[15]
print("the log:", crowded_log)
assert len(crowded_log) == 3, crowded_log
assert crowded_log[:2] == ["the store has no room for the changes of /lp %s now, so the command changed nothing" % (
    arguments) for arguments in ("deletegroup admin", "user Sam permission set golem.player")], crowded_log[:2]
assert crowded_log[2].endswith(
    "is %d changes behind, so the store refuses a change that must not wait for it" % BOUND), crowded_log[2]

## The queue

The queue hands out its actions oldest first, the ones the plugins queue and the ones pushed into
it alike, and a drain empties it. A second install replaces the handler and keeps the queue with
the actions in it.

In [ ]:
pushed = [
    {"kind": "broadcast", "text": "first"},
    {"kind": "command", "command": "say second"},
    {"kind": "message", "uuid": SAM["uuid"], "text": "third"},
]
steps, _ = script("queue", {
    "plugins": [{"kind": "echo", "name": "echo-a"}],
    "steps": [
        {"op": "push", "action": pushed[0]},
        {"op": "playerJoined", **SAM},
        {"op": "push", "action": pushed[1]},
        {"op": "install"},
        {"op": "push", "action": pushed[2]},
        {"op": "commandNames"},
        {"op": "drain"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[3]["queued"] == 3, "the second install lost the queue: %s" % steps[3]
assert steps[6]["decoded"] == [pushed[0], {"kind": "broadcast", "text": "echo-a playerJoined"}, pushed[1], pushed[2]], steps[6]
assert steps[7]["answer"] == [] and steps[7]["decoded"] == [], "the drain left actions behind: %s" % steps[7]
assert budgets_of(steps)["echo-a"]["calls"] == 0, "the second install kept the old runtime: %s" % budgets_of(steps)
print("drained in order:", [action.get("text") or action.get("command") for action in steps[6]["decoded"]])

## Tick statistics at given instants

`tick-stats` feeds ticks to a fresh window at instants that the cell chooses, so the expected
values are exact. The MSPT is the mean of the durations of the last 100 ticks in milliseconds.
The TPS is the number of intervals between those ticks divided by the time from the first to the
last, at most 20, and 0 while fewer than 2 ticks span no interval.

In [ ]:
def expected(ticks):
    window = ticks[-100:]
    mspt = sum(tick["nanos"] for tick in window) / len(window) / 1e6
    if len(window) < 2:
        return mspt, 0.0
    elapsed = (window[-1]["atMs"] - window[0]["atMs"]) / 1000
    return mspt, 20.0 if elapsed <= 0 else min(20.0, (len(window) - 1) / elapsed)


cases = {
    "one tick": [{"nanos": 7000000, "atMs": 0}],
    "every 100 ms": [{"nanos": 2000000, "atMs": i * 100.0} for i in range(10)],
    "every 25 ms, capped": [{"nanos": 1000000, "atMs": i * 25.0} for i in range(10)],
    "every 50 ms": [{"nanos": 49000000, "atMs": i * 50.0} for i in range(40)],
    "150 irregular ticks": [{"nanos": 1000000 * (i % 7 + 1) + 333 * i, "atMs": i * 50.0 + (i % 5) * 9.5 + (i // 30) * 40.0} for i in range(150)],
    "same instant": [{"nanos": 3000000, "atMs": 10.0}, {"nanos": 5000000, "atMs": 10.0}],
}
for name, ticks in cases.items():
    out, _ = devtool("tick-stats", write("ticks-%d.json" % len(ticks), ticks))
    assert out["window"] == 100 and out["maxTps"] == 20.0, out
    assert out["empty"] == {"ticks": 0, "mspt": 0.0, "tps": 0.0}, out["empty"]
    for count in range(1, len(ticks) + 1):
        mspt, tps = expected(ticks[:count])
        got = out["after"][count - 1]
        assert got["ticks"] == count, (name, count, got)
        assert abs(got["mspt"] - mspt) < 1e-9, (name, count, got, mspt)
        assert abs(got["tps"] - tps) < 1e-6, (name, count, got, tps)
    print("%-22s ticks %3d  mspt %8.4f  tps %7.4f" % (name, len(ticks), out["after"][-1]["mspt"], out["after"][-1]["tps"]))
assert out["after"][-1]["tps"] == 20.0

## Tick statistics on the wall clock

The native `tick` takes the instant of the call itself. The cell feeds ticks with a pause between
them, so the TPS must come out at or below the rate of the pauses and not far under it, then a
burst without pauses, which the cap holds at 20. A negative duration counts as 0.

In [ ]:
pauses = 6
steps, _ = script("wall", {
    "steps": [
        {"op": "stats"},
        {"op": "tick", "nanos": 3000000},
        {"op": "stats"},
        {"op": "tick", "nanos": 3000000, "repeat": pauses - 1, "sleepMs": tick_pause_ms},
        {"op": "stats"},
        {"op": "tick", "nanos": -5, "repeat": 30},
        {"op": "stats"},
    ],
})
before, one, paced, burst = (step["answer"] for step in steps if step["op"] == "stats")
rate = 1000 / tick_pause_ms
print("one tick:", one)
print("paced:", paced, "at most", rate)
print("burst:", burst)
assert before == {"ticks": 0, "mspt": 0.0, "tps": 0.0}, before
assert one == {"ticks": 1, "mspt": 3.0, "tps": 0.0}, one
assert paced["ticks"] == pauses and paced["mspt"] == 3.0, paced
assert 0.8 * rate <= paced["tps"] <= rate + 1e-9, "the paced TPS %.4f is not near %.1f" % (paced["tps"], rate)
assert burst["ticks"] == pauses + 30 and burst["tps"] == 20.0, burst
assert abs(burst["mspt"] - 3.0 * pauses / (pauses + 30)) < 1e-9, burst

## Pass values

With a handler that panics in every method, every native must answer the pass value of the
contract, nothing, false or an empty list, and log the panic, while the queue and the ticks go on.
Without a handler the natives answer the pass value as well. A null where the contract names a
string is logged and answered with the pass value.

In [ ]:
calls = [
    {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
    {"op": "serverStarted"},
    {"op": "tick", "nanos": 1000000},
    {"op": "playerJoined", **ALEX},
    {"op": "chat", **ALEX, "message": "spam"},
    {"op": "death", **ALEX, "cause": "minecraft:fall", **NETHER},
    {"op": "respawn", **ALEX, "dimension": "minecraft:overworld"},
    {"op": "playerLeft", **ALEX},
    {"op": "commandNames"},
    {"op": "command", "name": "golem", "arguments": "status"},
    {"op": "permission", **{"uuid": ALEX["uuid"]}, "node": "forge.use_entity_selectors"},
    {"op": "serverStopping"},
    {"op": "serverStopped"},
]
PASS = {"chat": False, "commandNames": [], "command": []}
extra = [{"op": "push", "action": {"kind": "command", "command": "stop"}}, {"op": "drain"}, {"op": "stats"}, {"op": "version"}]
for handler in ["panicking", "none"]:
    steps, log = script("pass-" + handler, {"handler": handler, "steps": calls + extra})
    for step in steps[:len(calls)]:
        assert step["answer"] == PASS.get(step["op"]), (handler, step)
    assert steps[-3]["decoded"] == [{"kind": "command", "command": "stop"}], steps[-3]
    assert steps[-2]["answer"]["ticks"] == 1, steps[-2]
    assert steps[-1]["answer"] == WORKSPACE["version"], steps[-1]
    panicked = [line for line in log if " panicked with " in line]
    print("%-9s pass values %d, panics logged %d" % (handler, len(calls), len(panicked)))
    assert len(panicked) == (len(calls) if handler == "panicking" else 0), log
    assert all(line.startswith("the native ") and line.endswith(", so it answers the pass value") for line in panicked), panicked

steps, log = script("nulls", {
    "budgetMs": budget_ms,
    "plugins": [{"kind": "echo", "name": "echo-a"}],
    "steps": [
        {"op": "chat", "name": "Alex", "message": "spam"},
        {"op": "playerJoined", "uuid": ALEX["uuid"]},
        {"op": "death", **ALEX, **NETHER},
        {"op": "death", **ALEX, "cause": "minecraft:fall"},
        {"op": "serverStarting", "minecraftVersion": "1.20.1"},
        {"op": "drain"},
    ],
})
assert steps[0]["answer"] is False and steps[5]["answer"] == [], steps
assert log == [
    "the native chat got null for uuid, so it answers the pass value",
    "the native playerJoined got null for name, so it answers the pass value",
    "the native death got null for cause, so it answers the pass value",
    "the native death got null for dimension, so it answers the pass value",
    "the native serverStarting got null for forgeVersion, so it answers the pass value",
], log

## Calls from other threads

Forge 47 calls `chat()` on a worker thread while the server thread runs its ticks, so a chat
message can arrive while a plugin still works on a tick. It must wait for the handler and reach
the plugins, not pass unseen. The cell runs a tick whose plugin sleeps, starts two chat messages
on threads of their own while it sleeps, and checks that the censor still cancels the one with
the word spam. A plugin that calls a native from inside the handler is different: that call
would wait for itself, so it answers the pass value at once, and the log names the first such call
of each native. Once the server thread is known, a call of another thread waits at most one
second for the handler: a chat message beside a tick whose plugin sleeps longer must pass unseen
after that second, before the tick ends, with one line on the log, while a chat message of the
server thread after the tick still reaches the censor.

In [ ]:
hold_ms = 5 * slow_ms
steps, log = script("threads", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "slow", "name": "sluggish", "on": "tick", "sleepMs": hold_ms},
        {"kind": "cancel", "name": "censor", "word": "spam"},
        {"kind": "reenter", "name": "loop", "on": "serverStarted"},
    ],
    "steps": [
        {"op": "together", "staggerMs": slow_ms, "steps": [
            {"op": "tick", "nanos": 1000000},
            {"op": "chat", **ALEX, "message": "buy spam now"},
            {"op": "chat", **SAM, "message": "hello"},
        ]},
        {"op": "serverStarted"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
tick, spam, hello = steps[0]["answer"]
for call in (tick, spam, hello):
    print("%-5s answered %-5s after %6.1f ms" % (call["op"], call["answer"], call["endMs"]))
assert tick["endMs"] >= hold_ms, tick
assert spam["answer"] is True, "a chat message beside a running tick escaped the plugins: %s" % spam
assert hello["answer"] is False, hello
assert spam["endMs"] >= 0.95 * hold_ms and hello["endMs"] >= 0.95 * hold_ms, "a chat message did not wait for the tick: %s" % steps[0]
assert broadcasts(steps[2]) == ["loop got 0 names"], steps[2]
calls = {name: row["calls"] for name, row in budgets_of(steps).items()}
print("calls:", calls)
assert calls == {"sluggish": 4, "censor": 4, "loop": 3}, "the cancel of the chat message must end its chain: %s" % calls
for line in log:
    print(line)
assert not any(" waited " in line for line in log), "a chat message stopped waiting for the tick: %s" % log
assert [line for line in log if "inside the handler" in line] == [
    "the native commandNames was called from inside the handler, so it answers the pass value, and later calls like "
    "it are not logged",
], log
assert len(log) == 2 and log[0].startswith("the plugin sluggish took "), log

OFF_THREAD_WAIT_MS = 1000
steps, log = script("threads-bound", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "slow", "name": "stuck", "on": "tick", "sleepMs": OFF_THREAD_WAIT_MS + 5 * slow_ms},
        {"kind": "cancel", "name": "censor", "word": "spam"},
    ],
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "together", "staggerMs": slow_ms, "steps": [
            {"op": "tick", "nanos": 1000000},
            {"op": "chat", **ALEX, "message": "buy spam now"},
        ]},
        {"op": "chat", **SAM, "message": "more spam"},
    ],
})
tick, spam = steps[1]["answer"]
for call in (tick, spam):
    print("%-5s answered %-5s after %6.1f ms" % (call["op"], call["answer"], call["endMs"]))
assert spam["answer"] is False, "a chat message waited for the handler longer than the bound: %s" % spam
assert slow_ms + 0.95 * OFF_THREAD_WAIT_MS <= spam["endMs"] < tick["endMs"], (spam, tick)
assert steps[2]["answer"] is True, "a chat message of the server thread did not wait for the handler: %s" % steps[2]
for line in log:
    print(line)
assert [line for line in log if " waited " in line] == [
    "the native chat waited 1000 ms for the handler, which another call holds, so it answers the pass value, 1 times "
    "so far"], log

## Verdict

The cell clears the temporary directory. A run that reaches it passed every section.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the 18 natives match okf/bridge.md, and the runtime, the core plugin, the queue the tick statistics and the calls of other threads answer as the wiki says")